# Observatório dos Leilões Minerais — projeto PP

Projeto independente. Cadastro: 02/10/2026. CFEM: até 06/2026.

Este caderno contém os scripts efetivamente utilizados e a base analítica mínima preservada, sem identificadores fiscais pessoais. O processamento completo exige os arquivos oficiais. Downloads atuais podem diferir do extrato do relatório. Os hashes abaixo permitem verificar a diferença. A execução gera arquivos internos com nomes de titulares recuperados do cadastro; não os publique sem revisão.

Os resultados são observacionais. Cenários de reserva e garantia não são previsões causais. A lista de 2021 é uma classificação histórica, não enquadramento jurídico automático na PNMCE.

In [ ]:
from pathlib import Path
import json, gzip, base64, hashlib, subprocess, sys, urllib.request
ROOT = Path('/content/leiloes-pp') if Path('/content').exists() else Path.cwd()/'leiloes-pp-reproducao'
for folder in ['src','data/raw','data/derived','data/complete']:
    (ROOT/folder).mkdir(parents=True, exist_ok=True)
print(ROOT)


In [ ]:
SCRIPTS = {'extract_scm.py': '"""Extrato independente PP. Identificadores pessoais não são exportados ao painel."""\nfrom pathlib import Path\nimport csv,io,zipfile,json,re,sys,collections,unicodedata\nsys.stdout.reconfigure(encoding=\'utf-8\')\nR=Path(__file__).resolve().parents[1]; D=json.loads((R/\'data/raw/analitico_preservado.json\').read_text(encoding=\'utf-8\'))\ndef proc(s):\n s=str(s).strip().replace(\'.\',\'\');m=re.fullmatch(r\'(\\d{1,6})/(\\d{4})\',s)\n if not m:return \'\'\n a=m[1].zfill(6);return a[:3]+\'.\'+a[3:]+\'/\'+m[2]\ndef fold(s):return \'\'.join(c for c in unicodedata.normalize(\'NFKD\',s) if not unicodedata.combining(c)).upper()\nz=zipfile.ZipFile(R/\'data/raw/scm.zip\')\ndef rows(n):\n path=next(x for x in z.namelist() if x.replace(\'\\\\\',\'/\').split(\'/\')[-1]==n+\'.txt\')\n with io.TextIOWrapper(z.open(path),encoding=\'cp1252\',errors=\'replace\',newline=\'\') as f:\n  yield from csv.DictReader(f,delimiter=\';\')\nevents={r[\'IDEvento\']:r[\'DSEvento\'] for r in rows(\'Evento\')}\ntypes={r[\'IDTipoAssociacao\']:r[\'DSTipoAssociacao\'] for r in rows(\'TipoAssociacao\')}\nphase={r[\'IDFaseProcesso\']:r[\'DSFaseProcesso\'] for r in rows(\'FaseProcesso\')}\norigins={r[\'origem\'] for r in D[\'rows\']}|{r[\'origin\'] for r in D[\'socialRound6\'][\'inventory\']}\noffers=list(csv.DictReader((R/\'data/raw/ResultadoRodadaDisponibilidade.csv\').open(encoding=\'utf-8-sig\'),delimiter=\';\'))\norigins|={proc(r[\'ProcessoMinerario\']) for r in offers};origins.discard(\'\')\ngraph=collections.defaultdict(list)\nfor r in rows(\'ProcessoAssociacao\'):\n a,b=proc(r[\'DSProcesso\']),proc(r[\'DSProcessoAssociado\'])\n if a and b and a!=b:graph[a].append({\'source\':a,\'target\':b,\'type\':r[\'IDTipoAssociacao\'],\'date\':r[\'DTAssociacao\'][:10],\'end\':r[\'DTDesassociacao\'][:10]})\nseen=set(origins);q=collections.deque(origins);links=[]\nwhile q:\n a=q.popleft()\n for e in graph[a]:\n  # Encadeamento explícito de disponibilidade, cessão parcial, desmembramento e mudança de regime.\n  if e[\'type\'] not in [\'2\',\'3\',\'8\',\'9\']:continue\n  links.append(e)\n  if e[\'target\'] not in seen:seen.add(e[\'target\']);q.append(e[\'target\'])\nprint(\'origens/processos/links\',len(origins),len(seen),len(links),flush=True)\nP={};stock=collections.Counter();stock_processes={}\nfor r in rows(\'Processo\'):\n stock[r[\'IDFaseProcesso\']]+=1\n p=proc(r[\'DSProcesso\'])\n if r[\'IDFaseProcesso\'] in [\'8\',\'15\']:stock_processes[p]=r\n if p in seen:P[p]=r\npeople=collections.defaultdict(list);ids=set()\nfor r in rows(\'ProcessoPessoa\'):\n p=proc(r[\'DSProcesso\'])\n if p in seen and r[\'IDTipoRelacao\'] in [\'1\',\'10\']:\n  people[p].append({\'id\':r[\'IDPessoa\'],\'role\':r[\'IDTipoRelacao\'],\'start\':r[\'DTInicioVigencia\'][:10],\'end\':r[\'DTFimVigencia\'][:10]});ids.add(r[\'IDPessoa\'])\nnames={}\nfor r in rows(\'Pessoa\'):\n if r[\'IDPessoa\'] in ids:names[r[\'IDPessoa\']]={\'name\':r[\'NMPessoa\'],\'type\':r[\'TPPessoa\']}\nev=collections.defaultdict(list);bad=0;total=0;stock_events=collections.defaultdict(list)\nstock_codes={k for k,v in events.items() if any(s in fold(v) for s in [\'DISPONIB\',\'RENUNCIA\',\'CADUCIDADE\',\'INDEF\',\'RECURSO\',\'DESONER\',\'S/EFEITO\',\'AREA LIVRE\'])}\nfor r in rows(\'ProcessoEvento\'):\n total+=1;p=proc(r[\'DSProcesso\'])\n if p not in seen and p not in stock_processes:continue\n if None in r or any(v is None for v in r.values()):bad+=1;continue\n if r[\'DTEvento\'][:10]>\'2026-10-02\':continue\n if p in stock_processes and r[\'IDEvento\'] in stock_codes:stock_events[p].append({\'code\':r[\'IDEvento\'],\'date\':r[\'DTEvento\'][:10]})\n if p not in seen:continue\n # Texto documental fica no extrato local, não é publicado sem revisão.\n ev[p].append({\'code\':r[\'IDEvento\'],\'date\':r[\'DTEvento\'][:10],\'note\':r[\'OBEvento\'],\'dou\':r[\'DSPublicacaoDOU\']})\n if total%2000000==0:print(\'eventos percorridos\',total,flush=True)\nsubnames={r[\'IDSubstancia\']:r[\'NMSubstancia\'] for r in rows(\'Substancia\')};subs=collections.defaultdict(list)\nfor r in rows(\'ProcessoSubstancia\'):\n p=proc(r[\'DSProcesso\'])\n if p in seen:subs[p].append({**r,\'name\':subnames.get(r[\'IDSubstancia\'],r[\'IDSubstancia\'])})\ntax=[]\nfor code,name in events.items():\n f=fold(name)\n if re.search(r\'\\bCESSAO\\b|\\bINCORP|\\bFUSAO\\b|\\bCAUSA MORTIS\\b|\\bCISAO\\b\',f):\n  stage=\'outro\'\n  if any(t in f for t in [\'ANULAD\',\'CANCELAD\',\'S/EFEITO\',\'NEGAD\',\'NEGA \',\'CASSAD\',\'DISTRATO\']):stage=\'reversao_ou_negativa\'\n  elif \'EFETIV\' in f or re.search(r\'AVERB(?:AD|$)\',f):stage=\'efetivacao\'\n  elif \'APROV\' in f or \'CADEIA SUCESS\' in f:stage=\'aprovacao\'\n  elif \'PROTOC\' in f:stage=\'pedido\'\n  family=\'cessao_parcial\' if \'PARC\' in f else \'incorporacao\' if \'INCORP\' in f else \'fusao\' if \'FUSAO\' in f else \'sucessao\' if \'CAUSA MORTIS\' in f else \'cisao\' if \'CISAO\' in f else \'cessao_total\'\n  tax.append({\'code\':code,\'label\':name,\'stage\':stage,\'family\':family})\nout={\'cutoff\':\'2026-10-02\',\'zip_dates\':{x.filename:str(x.date_time) for x in z.infolist()},\'origins\':sorted(origins),\'links\':links,\'processes\':P,\'owners\':people,\'people\':names,\'events\':ev,\'event_names\':events,\'taxonomy\':tax,\'substances\':subs,\'phase_names\':phase,\'association_types\':types,\'audit\':{\'malformed_relevant_events\':bad,\'all_events\':total,\'phase_counts\':dict(stock)}}\n(R/\'data/derived/stock_extract.json\').write_text(json.dumps({\'processes\':stock_processes,\'events\':stock_events,\'event_names\':events,\'phase_names\':phase},ensure_ascii=False,separators=(\',\',\':\')),encoding=\'utf-8\')\n(R/\'data/derived/scm_extract.json\').write_text(json.dumps(out,ensure_ascii=False,separators=(\',\',\':\')),encoding=\'utf-8\')\nprint(\'EXTRAÍDO\',len(P),sum(map(len,ev.values())),len(tax),out[\'audit\'])\n', 'statistics_pp.py': '"""Numpy implementations with explicit assumptions and deterministic checks."""\nimport numpy as np,math\n\ndef logistic(X,y,clusters=None,max_iter=150):\n X=np.asarray(X,float);y=np.asarray(y,float);b=np.zeros(X.shape[1]);ll=-np.inf\n for it in range(max_iter):\n  eta=np.clip(X@b,-35,35);p=1/(1+np.exp(-eta));w=np.maximum(p*(1-p),1e-9)\n  H=X.T@(X*w[:,None]);score=X.T@(y-p);step=np.linalg.solve(H+np.eye(len(b))*1e-9,score)\n  old=np.sum(y*eta-np.logaddexp(0,eta));scale=1\n  while scale>1e-7:\n   candidate=b+scale*step;v=X@candidate;new=np.sum(y*v-np.logaddexp(0,v))\n   if new>=old-1e-8:break\n   scale/=2\n  b=candidate\n  if np.max(np.abs(scale*step))<1e-8:break\n eta=X@b;p=1/(1+np.exp(-np.clip(eta,-35,35)));H=X.T@(X*(p*(1-p))[:,None]);bread=np.linalg.pinv(H)\n u=X*(y-p)[:,None]\n if clusters is not None:\n  keys,inv=np.unique(clusters,return_inverse=True);scores=np.zeros((len(keys),len(b)));np.add.at(scores,inv,u)\n  correction=len(keys)/max(1,len(keys)-1)*(len(y)-1)/(len(y)-len(b));cov=bread@(scores.T@scores)@bread*correction\n else:cov=bread@(u.T@u)@bread*len(y)/(len(y)-len(b))\n se=np.sqrt(np.maximum(np.diag(cov),0));z=np.divide(b,se,out=np.zeros_like(b),where=se>0)\n return {\'coef\':b.tolist(),\'se\':se.tolist(),\'or\':np.exp(b).tolist(),\'lo\':np.exp(b-1.96*se).tolist(),\'hi\':np.exp(b+1.96*se).tolist(),\'p\':[math.erfc(abs(x)/math.sqrt(2)) for x in z],\'n\':len(y),\'events\':int(sum(y)),\'iterations\':it+1,\'converged\':bool(np.max(np.abs(scale*step))<1e-6),\'score_max\':float(np.max(np.abs(X.T@(y-p)))),\'rank\':int(np.linalg.matrix_rank(X)),\'condition\':float(np.linalg.cond(H)),\'brier\':float(np.mean((p-y)**2)),\'probabilities\':p.tolist()}\n\ndef incidence(times,causes,horizons=(180,365,730,1095,1460,1825)):\n """Aalen-Johansen competing first events. cause 0=censor, 1=target, 2=exit."""\n t=np.asarray(times,int);c=np.asarray(causes,int);S=1.;F=0.;F2=0.;rows=[]\n all_sorted=np.sort(t);event_times=np.unique(t[c>0]);u1,c1=np.unique(t[c==1],return_counts=True);u2,c2=np.unique(t[c==2],return_counts=True);n1=dict(zip(u1,c1));n2=dict(zip(u2,c2))\n for x in event_times:\n  risk=int(len(t)-np.searchsorted(all_sorted,x));d1=int(n1.get(x,0));d2=int(n2.get(x,0))\n  F+=S*d1/risk;F2+=S*d2/risk;S*=1-(d1+d2)/risk\n  rows.append({\'day\':int(x),\'risk\':risk,\'events\':d1,\'exits\':d2,\'incidence\':F,\'exit_incidence\':F2,\'survival\':S})\n horizon=[]\n for h in horizons:\n  available=h<=max(t,default=0);v=[r for r in rows if r[\'day\']<=h];last=v[-1] if v else {\'incidence\':0.,\'exit_incidence\':0.,\'survival\':1.}\n  horizon.append({\'day\':h,\'available\':available,\'at_risk\':int(sum(t>=h)),\'incidence\':last[\'incidence\'] if available else None,\'exit_incidence\':last[\'exit_incidence\'] if available else None,\'survival\':last[\'survival\'] if available else None})\n return {\'curve\':rows,\'horizons\':horizon}\n\ndef cox(X,t,e,max_iter=60):\n """Cause-specific proportional hazards; Breslow ties, model-based covariance."""\n X=np.asarray(X,float);t=np.asarray(t,int);e=np.asarray(e,int)\n order=np.argsort(-t,kind=\'stable\');X=X[order];t=t[order];e=e[order];b=np.zeros(X.shape[1]);unique=np.unique(t[e==1]);idx=np.array([np.where(t>=v)[0][-1] for v in unique]);d=np.array([sum((t==v)&(e==1)) for v in unique]);xev=np.array([X[(t==v)&(e==1)].sum(0) for v in unique]);\n def calc(beta):\n  eta=X@beta;w=np.exp(np.clip(eta,-30,30));s0=np.cumsum(w)[idx];s1=np.cumsum(w[:,None]*X,axis=0)[idx];s2=np.cumsum(w[:,None,None]*X[:,:,None]*X[:,None,:],axis=0)[idx];m=s1/s0[:,None]\n  ll=float((X[e==1]@beta).sum()-np.sum(d*np.log(s0)));g=(xev-d[:,None]*m).sum(0);H=np.sum(d[:,None,None]*(s2/s0[:,None,None]-m[:,:,None]*m[:,None,:]),axis=0)\n  return ll,g,H\n for it in range(max_iter):\n  ll,g,H=calc(b);step=np.linalg.solve(H+np.eye(len(b))*1e-9,g);scale=1\n  while scale>1e-7 and calc(b+scale*step)[0]<ll-1e-9:scale/=2\n  b+=scale*step\n  if np.max(abs(scale*step))<1e-8:break\n ll,g,H=calc(b);se=np.sqrt(np.maximum(0,np.diag(np.linalg.pinv(H))))\n return {\'coef\':b.tolist(),\'se\':se.tolist(),\'hr\':np.exp(b).tolist(),\'lo\':np.exp(b-1.96*se).tolist(),\'hi\':np.exp(b+1.96*se).tolist(),\'n\':len(t),\'events\':int(sum(e)),\'converged\':bool(np.max(abs(scale*step))<1e-6),\'score_max\':float(max(abs(g))),\'iterations\':it+1,\'covariance\':\'Model-based inverse information; observational association, Breslow ties.\'}\n\ndef check():\n # Known intercept-only MLE.\n fit=logistic(np.ones((100,1)),np.r_[np.ones(30),np.zeros(70)])\n assert abs(fit[\'coef\'][0]-math.log(.3/.7))<1e-6\n # First event and competing exit among four units: each cumulative incidence is .25.\n a=incidence([1,2,3,4],[1,2,0,0],horizons=[2])[\'horizons\'][0]\n assert abs(a[\'incidence\']-.25)<1e-9 and abs(a[\'exit_incidence\']-.25)<1e-9\n assert abs(a[\'survival\']-.5)<1e-9\n # Numerical gradient/Hessian validation for Cox by a simulated known direction.\n rng=np.random.default_rng(721);x=rng.normal(size=(600,1));t=np.round(rng.exponential(100/np.exp(.7*x[:,0]))).astype(int)+1;c=np.full(600,200);m=cox(x,np.minimum(t,c),(t<=c).astype(int));assert .35<m[\'coef\'][0]<1.1 and m[\'converged\']\n return {\'logistic_intercept\':\'passed\',\'competing_risk_identity\':\'passed\',\'cox_simulated_direction\':\'passed\'}\n\nif __name__==\'__main__\':print(check())\n', 'complete_analysis.py': '"""Executed PP analysis; all empirical outputs and scenarios are separately labeled."""\nfrom pathlib import Path\nimport json,csv,collections,datetime,re,unicodedata,statistics,zipfile,io,hashlib,math\nimport numpy as np\nfrom statistics_pp import logistic,incidence,cox,check\nR=Path(__file__).resolve().parents[1];O=R/\'data/complete\';O.mkdir(exist_ok=True)\ndef save(name,obj):(O/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding=\'utf8\')\ndef csvsave(name,rows):\n if not rows:return\n with (O/name).open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n  w=csv.DictWriter(f,fieldnames=list(rows[0]),delimiter=\';\');w.writeheader();w.writerows(rows)\ndef fold(s):return \'\'.join(x for x in unicodedata.normalize(\'NFKD\',str(s)) if not unicodedata.combining(x)).upper()\ndef proc(s):\n m=re.fullmatch(r\'(\\d{1,6})/(\\d{4})\',str(s).strip().replace(\'.\',\'\'))\n if not m:return \'\'\n a=m[1].zfill(6);return a[:3]+\'.\'+a[3:]+\'/\'+m[2]\ndef days(a,b):return (datetime.date.fromisoformat(b[:10])-datetime.date.fromisoformat(a[:10])).days\ndef date(x):return bool(re.fullmatch(r\'\\d{4}-\\d{2}-\\d{2}\',x or \'\'))\ndef money(s):return float(str(s).replace(\'.\',\'\').replace(\',\',\'.\')) if s else 0.\ndef readcsv(n):return list(csv.DictReader((R/\'data/raw\'/n).open(encoding=\'utf-8-sig\'),delimiter=\';\'))\nD=json.loads((R/\'data/raw/analitico_preservado.json\').read_text(encoding=\'utf8\'));S=json.loads((R/\'data/derived/scm_extract.json\').read_text(encoding=\'utf8\'))\ncut=\'2026-10-02\';exclusive=\'2026-10-03\';ref={r[\'origem\']:r for r in D[\'rows\']};offers=readcsv(\'ResultadoRodadaDisponibilidade.csv\');social=readcsv(\'ResultadoAvaliacaoSocial.csv\');stock=readcsv(\'EstoqueAreas.csv\')\noffer_groups=collections.defaultdict(list)\nfor x in offers:offer_groups[(int(x[\'Rodada\']),x[\'NumeroArea\'],proc(x[\'ProcessoMinerario\']))].append(x)\nref.update(D[\'cycleResolution\'][\'rows\'])\ncut_by_round={int(a[\'round\']):a[\'homologation\'] for a in D[\'areaScenario\'][\'areas\']}\n\n# A historical strategic-mineral proxy, explicitly not a PNMCE designation.\ngroups={\'I\':{\'enxofre\',\'fosfato\',\'potássio\',\'molibdênio\'},\'II\':{\'cobalto\',\'cobre\',\'estanho\',\'grafita\',\'platina\',\'lítio\',\'nióbio\',\'níquel\',\'silício\',\'tálio\',\'tântalo\',\'terras raras\',\'titânio\',\'tungstênio\',\'urânio\',\'vanádio\'},\'III\':{\'alumínio\',\'cobre\',\'ferro\',\'grafita\',\'ouro\',\'manganês\',\'nióbio\',\'urânio\'}}\ntokens={\n \'enxofre\':[\'ENXOFRE\'],\'fosfato\':[\'FOSFATO\',\'FOSFORITA\',\'APATITA\'],\'potássio\':[\'POTASSIO\',\'SILVITA\',\'CARNALITA\'],\'molibdênio\':[\'MOLIBDENIO\',\'MOLIBDENITA\'],\n \'cobalto\':[\'COBALTO\'],\'cobre\':[\'COBRE\'],\'estanho\':[\'ESTANHO\',\'CASSITERITA\'],\'grafita\':[\'GRAFITA\'],\'platina\':[\'PLATINA\',\'PALADIO\',\'RODIO\',\'RUTENIO\',\'IRIDIO\',\'OSMIO\'],\n \'lítio\':[\'LITIO\',\'ESPODUMENIO\',\'PETALITA\',\'LEPIDOLITA\'],\'nióbio\':[\'NIOBIO\',\'PIROCLORO\'],\'níquel\':[\'NIQUEL\'],\'silício\':[\'SILICIO\'],\'tálio\':[\'TALIO\'],\n \'tântalo\':[\'TANTALO\',\'TANTALITA\'],\'terras raras\':[\'TERRAS RARAS\',\'MONAZITA\',\'XENOTIMA\'],\'titânio\':[\'TITANIO\',\'ILMENITA\',\'RUTILO\'],\'tungstênio\':[\'TUNGSTENIO\',\'SCHEELITA\',\'WOLFRAMITA\'],\n \'urânio\':[\'URANIO\'],\'vanádio\':[\'VANADIO\'],\'alumínio\':[\'ALUMINIO\',\'BAUXITA\'],\'ferro\':[\'FERRO\',\'HEMATITA\',\'MAGNETITA\'],\'ouro\':[\'OURO\'],\'manganês\':[\'MANGANES\']}\naudit_sub={}\ndef classify(names):\n hits=set();amb=set()\n for name in names:\n  f=fold(name);local=set()\n  for mineral,terms in tokens.items():\n   if any(re.search(r\'\\b\'+t+r\'\\b\',f) for t in terms):local.add(mineral)\n  if \'COLUMBITA\' in f:local|={\'nióbio\'};amb.add(\'columbita pode conter tântalo; apenas nióbio incluído\')\n  if \'QUARTZO\' in f or \'QUARTZITO\' in f or \'AREIA\' in f:amb.add(\'material silicoso não incluído automaticamente como silício\')\n  audit_sub[name]={\'original\':name,\'mapped\':\' | \'.join(sorted(local)),\'notes\':\' | \'.join(sorted(amb))}\n  hits|=local\n return sorted(hits),sorted(amb)\ndef substances(p,t=None):\n records=S[\'substances\'].get(p,[])\n return sorted(set(x[\'name\'] for x in records if not t or ((not x[\'DTInicioVigencia\'] or x[\'DTInicioVigencia\'][:10]<=t) and (not x[\'DTFimVigencia\'] or x[\'DTFimVigencia\'][:10]>=t))))\n\n# Human-readable operational event codebook. Unclassified codes are retained, not guessed.\ncodebook=[];tax={}\nfor x in S[\'taxonomy\']:\n code=x[\'code\'];label=x[\'label\'];f=fold(label);family=x[\'family\'];stage=x[\'stage\'];reason=\'Descrição literal do evento.\'\n if code in [\'119\',\'339\',\'108\']:family=\'misto_incorporacao_cessao\';reason=\'Descrição mistura incorporação e cessão; excluído da definição estrita.\'\n if any(t in f for t in [\'EMOLUMENT\',\'PROVENIENTE\',\'ASSENTIMENTO\',\'CDN\',\'EXIGENCIA\',\'INTIMACAO\',\'DOCUMENTO DIVERSO\',\'COMPROVANTE\']):stage=\'contexto\';reason=\'Ato auxiliar; não prova pedido ou efetivação da cessão.\'\n elif \'DISTRATO\' in f:stage=\'pedido_distrato\';reason=\'Solicitação de desfazimento não equivale a desfazimento efetivo.\'\n elif any(t in f for t in [\'ANULAD\',\'CANCELAD\',\'CASSAD\',\'S/EFEITO\']):stage=\'reversao_candidata\';reason=\'Conferir o ato atingido. Reversão de negativa não equivale a anulação da cessão.\'\n elif any(t in f for t in [\'NEGAD\',\'NEGA \']):stage=\'negativa\';reason=\'Não representa transferência concluída.\'\n item={**x,\'family\':family,\'stage\':stage,\'justification\':reason};codebook.append(item);tax[code]=item\nstrict={k for k,x in tax.items() if x[\'stage\']==\'efetivacao\' and x[\'family\'] in [\'cessao_total\',\'cessao_parcial\']}\nreturn_codes={\'2463\',\'2464\',\'2466\',\'2471\',\'2472\',\'2473\'}\npayment_codes={\'2463\',\'2471\',\'2778\'}\nnegative_payment={\'2464\',\'2472\',\'2473\'}\nfor code in sorted(return_codes|payment_codes|negative_payment,key=int):codebook.append({\'code\':code,\'label\':S[\'event_names\'][code],\'family\':\'pagamento_continuidade\',\'stage\':\'interrupcao\' if code in return_codes else \'pagamento_registrado\',\'justification\':\'Evidência cadastral; não substitui conciliação bancária. Evento limitado ao ciclo correspondente.\'})\nsave(\'event_codebook.json\',codebook)\nedges=collections.defaultdict(list)\nfor e in S[\'links\']:edges[e[\'source\']].append(e)\ncycles=collections.defaultdict(list)\nfor a in D[\'areaScenario\'][\'areas\']:cycles[a[\'origin\']].append(a[\'homologation\'])\naudits=[];all_events=[];trajectories=[];rows=[];seen_events=set()\ncess_csv=collections.defaultdict(set)\nwith (R/\'data/raw/Cessoes_de_Direitos.csv\').open(encoding=\'cp1252\',newline=\'\') as f:\n for x in csv.DictReader(f):\n  try:t=datetime.datetime.strptime(x[\'Data da Cessão\'],\'%d/%m/%Y\').date().isoformat()\n  except ValueError:continue\n  cess_csv[proc(x[\'Processo\'])].add(t)\n\nfor a in D[\'areaScenario\'][\'areas\']:\n start=a[\'homologation\'];end=min([t for t in cycles[a[\'origin\']] if t>start]+[exclusive]);root=a[\'origin\'];rawref=ref[root]\n intervals={root:[(start,end)]};queue=collections.deque([(root,start,end)]);used=[];visited=set()\n while queue:\n  p,lo,hi=queue.popleft()\n  if (p,lo,hi) in visited:continue\n  visited.add((p,lo,hi))\n  for link in edges[p]:\n   t=link[\'date\'];stop=min(hi,link[\'end\'] or hi)\n   if not date(t) or not lo<=t<stop:continue\n   child=link[\'target\'];stop=min([v for v in cycles.get(child,[]) if v>=t]+[stop]);interval=(t,stop)\n   if t>=stop:continue\n   if child==root:continue\n   used.append(link)\n   if interval not in intervals.get(child,[]):intervals.setdefault(child,[]).append(interval);queue.append((child,t,stop))\n events=[]\n for p,windows in intervals.items():\n  for e in S[\'events\'].get(p,[]):\n   if any(lo<=e[\'date\']<hi for lo,hi in windows):events.append({**e,\'process\':p,\'label\':S[\'event_names\'].get(e[\'code\'],e[\'code\']),\'days\':days(start,e[\'date\'])})\n exits=[e for e in events if e[\'process\']==root and e[\'code\'] in return_codes]\n exit_day=min((e[\'days\'] for e in exits),default=None)\n histories=[e for e in events if e[\'code\'] in strict]\n unique={ (e[\'process\'],e[\'date\'],tax[e[\'code\']][\'family\']):e for e in histories};histories=sorted(unique.values(),key=lambda e:(e[\'date\'],e[\'process\']))\n strict_events=[e for e in histories if exit_day is None or e[\'days\']<exit_day]\n successor=[e for e in strict_events if e[\'process\']!=root]\n root_only=[e for e in strict_events if e[\'process\']==root]\n reversals=[e for e in events if e[\'code\'] in tax and tax[e[\'code\']][\'stage\']==\'reversao_candidata\']\n sensitive=[e for e in strict_events if not any(z[\'process\']==e[\'process\'] and z[\'date\']>=e[\'date\'] for z in reversals)]\n requests=[e for e in events if e[\'code\'] in tax and tax[e[\'code\']][\'stage\']==\'pedido\' and tax[e[\'code\']][\'family\'] in [\'cessao_total\',\'cessao_parcial\']]\n approvals=[e for e in events if e[\'code\'] in tax and tax[e[\'code\']][\'stage\']==\'aprovacao\' and tax[e[\'code\']][\'family\'] in [\'cessao_total\',\'cessao_parcial\']]\n denials=[e for e in events if e[\'code\'] in tax and tax[e[\'code\']][\'stage\']==\'negativa\']\n corporate=[e for e in events if e[\'code\'] in tax and tax[e[\'code\']][\'stage\']==\'efetivacao\' and e[\'code\'] not in strict]\n owners=[]\n for p,windows in intervals.items():\n  people=[x for x in S[\'owners\'].get(p,[]) if x[\'role\']==\'1\' and date(x[\'start\'])]\n  for x in people:\n   if not any(lo<x[\'start\']<hi for lo,hi in windows):continue\n   prior=[y for y in people if y[\'id\']!=x[\'id\'] and y[\'start\']<x[\'start\'] and date(y[\'end\']) and y[\'end\']<=x[\'start\']]\n   if prior:\n    prev=max(prior,key=lambda y:y[\'end\']);owners.append({\'process\':p,\'date\':x[\'start\'],\'from\':prev[\'id\'],\'to\':x[\'id\'],\'gap\':days(prev[\'end\'],x[\'start\'])})\n first=strict_events[0][\'days\'] if strict_events else None;follow=days(start,min(end,cut));cause=1 if first is not None and (exit_day is None or first<exit_day) else 2 if exit_day is not None else 0;time=first if cause==1 else exit_day if cause==2 else follow\n names=substances(root,start);minerals,ambiguity=classify(names);fallback=not names\n if fallback:names=[x.strip() for x in rawref[\'substancia\'].split(\';\') if x.strip()];minerals,ambiguity=classify(names)\n offer=offer_groups.get((a[\'round\'],str(a[\'area\']),root),[]);winners=sorted(set(x[\'NomeVencedor\'] for x in offer if x[\'Situacao\']==\'Arrematada\'))\n raw_area=rawref.get(\'ha\',\'\');area=money(raw_area) if raw_area else 0\n post_titles=[e for e in events if e[\'code\'] in {\'201\',\'323\',\'400\',\'625\',\'624\',\'705\',\'730\',\'2210\'} and (first is not None and e[\'days\']>first)]\n confirmed=[e for e in events if e[\'code\'] in payment_codes];unpaid_new=[e for e in events if e[\'code\'] in negative_payment]\n r={\'key\':a[\'key\'],\'origin\':root,\'round\':a[\'round\'],\'area\':str(a[\'area\']),\'start\':start,\'end\':end,\'bid\':a[\'cents\']/100,\'unpaid\':int(a[\'unpaid\']),\'unpaid_current\':int(bool(unpaid_new)),\'payment_record\':int(bool(confirmed)),\'payment_conflict\':int(bool(confirmed) and bool(unpaid_new)),\'winner\':winners[0] if len(winners)==1 else \'\', \'state\':rawref[\'uf\'],\'city\':rawref[\'municipio\'],\'regime\':rawref[\'regime\'],\'ha\':area,\'minerals\':\' | \'.join(minerals),\'strategic\':int(bool(minerals)),\'substance_fallback\':fallback,\'substances\':\'; \'.join(names),\'nodes\':len(intervals),\'request\':int(bool(requests)),\'approval\':int(bool(approvals)),\'denial\':int(bool(denials)),\'cession\':int(bool(strict_events)),\'cessions\':len(strict_events),\'total_cessions\':sum(tax[e[\'code\']][\'family\']==\'cessao_total\' for e in strict_events),\'partial_cessions\':sum(tax[e[\'code\']][\'family\']==\'cessao_parcial\' for e in strict_events),\'reversal\':int(bool(reversals)),\'cession_no_later_reversal\':int(bool(sensitive)),\'cession_successor_only\':int(bool(successor)),\'cession_root_only\':int(bool(root_only) and not successor),\'cession_excluding_unpaid\':int(bool(strict_events) and not a[\'unpaid\']),\'corporate\':int(bool(corporate)),\'holder\':int(bool(owners)),\'holder_n\':len({(x[\'process\'],x[\'date\'],x[\'to\']) for x in owners}),\'first_days\':first,\'exit_days\':exit_day,\'time\':time,\'cause\':cause,\'followup\':follow,\'first_cfem\':rawref.get(\'primeiro_depois\'),\'cfem\':(rawref.get(\'depois\') or {}).get(\'reais\',0),\'fiscal_eligible\':bool(rawref.get(\'depois\') is not None and D[\'businessOutcomes\'][\'origins\'][root][\'state\'] not in [\'returned_unpaid\',\'free_unpaid\',\'paid_without_request\']),\'post_cession_title\':bool(post_titles),\'csv_cession_match\':int(any(e[\'date\'] in cess_csv[e[\'process\']] for e in strict_events))}\n rows.append(r)\n if strict_events or corporate or owners:\n  enriched=[]\n  relevant=[e for e in events if e[\'code\'] in tax or e[\'code\'] in return_codes|payment_codes|{\'201\',\'323\',\'400\',\'625\',\'624\',\'705\',\'730\',\'2210\'}]\n  for e in relevant:\n   entry={k:e[k] for k in [\'process\',\'date\',\'code\',\'label\',\'days\']}\n   entry[\'stage\']=tax.get(e[\'code\'],{}).get(\'stage\',\'marco\');entry[\'family\']=tax.get(e[\'code\'],{}).get(\'family\',\'marco\');entry[\'reversal_candidate\']=any(z[\'process\']==e[\'process\'] and z[\'date\']>=e[\'date\'] for z in reversals)\n   enriched.append(entry)\n  trajectories.append({\'summary\':r,\'links\':used,\'events\':sorted(enriched,key=lambda x:(x[\'date\'],x[\'process\'])),\'holders\':[{**x,\'from_name\':S[\'people\'].get(x[\'from\'],{}).get(\'name\',\'Não identificado\'),\'to_name\':S[\'people\'].get(x[\'to\'],{}).get(\'name\',\'Não identificado\')} for x in owners]})\n for e in strict_events:\n  all_events.append({\'key\':a[\'key\'],\'origin\':root,\'round\':a[\'round\'],\'process\':e[\'process\'],\'date\':e[\'date\'],\'code\':e[\'code\'],\'family\':tax[e[\'code\']][\'family\'],\'days\':e[\'days\'],\'matched_csv\':e[\'date\'] in cess_csv[e[\'process\']]})\nprint(\'Trajectories built\',len(rows),flush=True)\ncsvsave(\'award_analysis.csv\',rows);csvsave(\'cession_events.csv\',all_events);save(\'trajectories_internal.json\',trajectories)\n\ndef summary(rs):\n event=[r[\'first_days\'] for r in rs if r[\'first_days\'] is not None]\n return {\'n\':len(rs),\'bid\':sum(r[\'bid\'] for r in rs),\'unpaid\':sum(r[\'unpaid\'] for r in rs),\'unpaid_value\':sum(r[\'bid\'] for r in rs if r[\'unpaid\']),\'cession\':sum(r[\'cession\'] for r in rs),\'cessions\':sum(r[\'cessions\'] for r in rs),\'recurrent\':sum(r[\'cessions\']>1 for r in rs),\'total\':sum(r[\'total_cessions\'] for r in rs),\'partial\':sum(r[\'partial_cessions\'] for r in rs),\'request\':sum(r[\'request\'] for r in rs),\'approval\':sum(r[\'approval\'] for r in rs),\'denial\':sum(r[\'denial\'] for r in rs),\'reversal\':sum(r[\'reversal\'] for r in rs),\'holder\':sum(r[\'holder\'] for r in rs),\'corporate\':sum(r[\'corporate\'] for r in rs),\'observed_median\':float(np.median(event)) if event else None,\'under180\':sum(x<=180 for x in event),\'under365\':sum(x<=365 for x in event),\'eligible\':sum(r[\'fiscal_eligible\'] for r in rs),\'cfem_positive\':sum(r[\'cfem\']>0 and r[\'fiscal_eligible\'] for r in rs),\'payment_record\':sum(r[\'payment_record\'] for r in rs),\'payment_conflict\':sum(r[\'payment_conflict\'] for r in rs)}\nresults={\'overall\':summary(rows),\'by_round\':{str(k):summary([r for r in rows if r[\'round\']==k]) for k in sorted({r[\'round\'] for r in rows})},\'by_regime\':{k:summary([r for r in rows if r[\'regime\']==k]) for k in sorted({r[\'regime\'] for r in rows})},\'sensitivity\':{k:sum(r[k] for r in rows) for k in [\'cession\',\'cession_no_later_reversal\',\'cession_successor_only\',\'cession_root_only\',\'cession_excluding_unpaid\',\'holder\',\'csv_cession_match\',\'unpaid_current\',\'payment_record\',\'payment_conflict\']},\'independent_event_keys\':len({(e[\'process\'],e[\'date\'],e[\'family\']) for e in all_events}),\'overlap_holder_cession\':dict(collections.Counter(f\'cessao={r["cession"]};titular={r["holder"]}\' for r in rows)),\'code_counts\':dict(collections.Counter(e[\'code\'] for e in all_events))}\naj=incidence([r[\'time\'] for r in rows],[r[\'cause\'] for r in rows]);results[\'incidence\']=aj\nresults[\'incidence_rounds\']={str(k):incidence([r[\'time\'] for r in rows if r[\'round\']==k],[r[\'cause\'] for r in rows if r[\'round\']==k]) for k in sorted({r[\'round\'] for r in rows})}\n# Cluster bootstrap of first-event incidence at 365 and 730 days.\nrng=np.random.default_rng(20261003);cluster=collections.defaultdict(list)\nfor r in rows:cluster[r[\'origin\']].append(r)\nck=list(cluster);boot=[]\nfor b in range(200):\n sampled=[r for k in rng.choice(ck,len(ck),replace=True) for r in cluster[k]];v=incidence([r[\'time\'] for r in sampled],[r[\'cause\'] for r in sampled],horizons=[365,730])[\'horizons\'];boot.append([x[\'incidence\'] for x in v])\nresults[\'incidence_ci\']={str(h):{\'lo\':float(np.quantile(np.array(boot)[:,i],.025)),\'hi\':float(np.quantile(np.array(boot)[:,i],.975)),\'replicates\':200} for i,h in enumerate([365,730])}\nprint(\'Incidence calculated\',flush=True)\n\n# Association models. No look-ahead covariates such as later titles.\nmodelrows=[r for r in rows if r[\'round\']!=1 and r[\'ha\']>0]\nnames=[\'Intercepto\',\'Log do lance\',\'Log da área\',\'Regime lavra\',\'Substância estratégica proxy\',\'Rodada 3\',\'Rodada 4\',\'Rodada 5\',\'Rodada 8\']\nX=np.array([[1,math.log1p(r[\'bid\']),math.log1p(r[\'ha\']),int(\'Lavra\' in r[\'regime\']),r[\'strategic\'],*[int(r[\'round\']==k) for k in [3,4,5,8]]] for r in modelrows]);Y=np.array([r[\'unpaid\'] for r in modelrows]);cl=np.array([r[\'origin\'] for r in modelrows])\nfit=logistic(X,Y,cl);pred=fit.pop(\'probabilities\');fit[\'variables\']=names;fit[\'excluded_round1\']=\'Nenhum sinal de não pagamento; exclusão evita separação completa.\';results[\'logistic_nonpayment\']=fit\nthreshold=np.quantile([r[\'bid\'] for r in modelrows],.99);keep=np.array([r[\'bid\']<=threshold for r in modelrows]);sens=logistic(X[keep],Y[keep],cl[keep]);sens.pop(\'probabilities\');sens[\'variables\']=names;sens[\'threshold\']=threshold;results[\'logistic_without_top1\']=sens\ncoxfit=cox(X[:,1:],[r[\'time\'] for r in modelrows],[r[\'cause\']==1 for r in modelrows]);coxfit[\'variables\']=names[1:];results[\'cox_cession\']=coxfit\n# Complete-followup comparison to avoid maturity bias.\nfixed=[r for r in rows if r[\'followup\']>=730];results[\'fixed_730\']={str(k):{\'n\':len(a:=[r for r in fixed if r[\'round\']==k]),\'cession\':sum(r[\'first_days\'] is not None and r[\'first_days\']<=730 for r in a),\'exit\':sum(r[\'exit_days\'] is not None and r[\'exit_days\']<=730 for r in a)} for k in sorted({r[\'round\'] for r in fixed})}\n# Payment concentration, descriptive bands, winner portfolios without exposing personal IDs.\nbands=[0,1000,10000,100000,1000000,float(\'inf\')];results[\'bid_bands\']=[]\nfor lo,hi in zip(bands,bands[1:]):\n rs=[r for r in rows if lo<=r[\'bid\']<hi];results[\'bid_bands\'].append({\'lo\':lo,\'hi\':None if math.isinf(hi) else hi,**summary(rs)})\nunpaid=sorted([r[\'bid\'] for r in rows if r[\'unpaid\']],reverse=True);results[\'concentration\']={str(k):sum(unpaid[:k])/sum(unpaid) for k in [1,5,10,20,50,100]}\nwinnergroups=collections.defaultdict(list)\nfor r in rows:\n if r[\'winner\']:winnergroups[r[\'winner\']].append(r)\nresults[\'winner_portfolios\']=[{\'size\':label,**summary([r for rs in winnergroups.values() if lo<=len(rs)<=hi for r in rs])} for label,lo,hi in [(\'1\',1,1),(\'2 a 5\',2,5),(\'6 a 20\',6,20),(\'21 ou mais\',21,99999)]]\n# Historical screen of minimum bid; explicitly not behavioral prediction.\nresults[\'minimum_screens\']=[]\nfor reserve in [0,1000,5000,10000,25000,50000,100000,250000]:\n kept=[r for r in rows if r[\'bid\']>=reserve];excluded=[r for r in rows if r[\'bid\']<reserve];results[\'minimum_screens\'].append({\'reserve\':reserve,\'retained\':len(kept),\'excluded\':len(excluded),\'excluded_value\':sum(r[\'bid\'] for r in excluded),\'excluded_signal\':sum(r[\'unpaid\'] for r in excluded),\'excluded_signal_value\':sum(r[\'bid\'] for r in excluded if r[\'unpaid\']),\'remaining_signal_value\':sum(r[\'bid\'] for r in kept if r[\'unpaid\'])})\n# Stylized calibrated reduced-form sensitivity. NOT equilibrium or causal estimates.\nbase_rate=sum(Y)/len(Y);results[\'incentive_scenarios\']=[]\nfor reserve in [0,10000,50000,100000]:\n kept=[r for r in rows if r[\'bid\']>=reserve];nominal=sum(r[\'bid\'] for r in kept);bad=sum(r[\'bid\'] for r in kept if r[\'unpaid\'])\n for guarantee in [0,.02,.05,.10]:\n  for elasticity in [.5,1.5,3.]:\n   entry=math.exp(-elasticity*guarantee)\n   for efficacy in [0,.25,.5]:\n    # Recovery on guarantee intentionally excluded; require lawful execution evidence.\n    proxy=entry*(nominal-bad*(1-efficacy));results[\'incentive_scenarios\'].append({\'reserve\':reserve,\'guarantee\':guarantee,\'entry_elasticity\':elasticity,\'default_reduction_assumed\':efficacy,\'participation_factor\':entry,\'commitment_proxy\':proxy,\'guarantee_locked\':entry*nominal*guarantee,\'break_even_default_reduction\':max(0,(nominal-bad)*(1/entry-1)/bad) if bad else None,\'label\':\'Cenário hipotético; complemento do sinal não é pagamento confirmado.\'})\n\n# Strategic proxy on the financial cohorts and every round of offers.\nresults[\'strategic_comparison\']={\'proxy\':summary([r for r in rows if r[\'strategic\']]),\'other\':summary([r for r in rows if not r[\'strategic\']])}\nresults[\'strategic_minerals\']={m:summary([r for r in rows if m in r[\'minerals\'].split(\' | \')]) for m in tokens}\nresults[\'strategic_rounds\']={str(k):summary([r for r in rows if r[\'round\']==k and r[\'strategic\']]) for k in sorted({r[\'round\'] for r in rows})}\noffered=[]\nfor (round_,area,p),rs in offer_groups.items():\n names=substances(p);mins,amb=classify(names);statuses=sorted(set(x[\'Situacao\'] for x in rs));offered.append({\'round\':round_,\'area\':area,\'process\':p,\'statuses\':\' | \'.join(statuses),\'substances\':\'; \'.join(names),\'minerals\':\' | \'.join(mins),\'proxy\':int(bool(mins)),\'coverage\':bool(names),\'basis\':\'Cadastro 02/10/2026; retrospectivo, não prova informação disponível no edital.\'})\nfor x in social:\n p=proc(x[\'ProcessoMinerario\']);mins,amb=classify(substances(p));offered.append({\'round\':6,\'area\':x[\'NumeroArea\'],\'process\':p,\'statuses\':x[\'Situacao\'],\'substances\':\'; \'.join(substances(p)),\'minerals\':\' | \'.join(mins),\'proxy\':int(bool(mins)),\'coverage\':bool(substances(p)),\'basis\':\'Cadastro 02/10/2026; avaliação social.\'})\n# Dedup social records by area/process, preserving requested as terminal status.\noffered_unique={}\nfor x in offered:\n key=(x[\'round\'],x[\'area\'],x[\'process\'])\n if key not in offered_unique or x[\'statuses\']==\'Requerido\':offered_unique[key]=x\noffered=list(offered_unique.values());csvsave(\'all_rounds_minerals.csv\',offered);csvsave(\'substance_mapping.csv\',list(audit_sub.values()));results[\'all_rounds\']={str(k):{\'areas\':len(rs:=[x for x in offered if x[\'round\']==k]),\'proxy\':sum(x[\'proxy\'] for x in rs),\'without_substance\':sum(not x[\'coverage\'] for x in rs),\'statuses\':dict(collections.Counter(x[\'statuses\'] for x in rs))} for k in range(1,9)}\nresults[\'mineral_proxy_note\']=\'Resolução SGM/MME 2/2021, com minerais portadores explicitamente mapeados. Não equivale a enquadramento legal PNMCE nem a reserva comprovada.\'\nresults[\'checks\']=check();results[\'parameters\']={\'cutoff\':cut,\'fiscal_cutoff\':\'2026-06\',\'bootstrap_seed\':20261003,\'bootstrap_replicates\':200,\'event_units\':\'processo/data/família dentro de ciclo e intervalo de associação\',\'cox_ties\':\'Breslow\',\'logistic_covariance\':\'sandwich agrupado por origem\',\'first_event_competition\':\'retorno/liberação explícita da origem nos códigos 2463 2464 2466 2471 2472 2473\',\'unpaid_model_outcome\':\'sinal preservado, não conciliação financeira\',\'mineral_asof\':\'data da homologação, fallback explicitado; todas ofertas usam retrato 02/10/2026\'}\nsave(\'results.json\',results)\nprint(json.dumps({\'overall\':results[\'overall\'],\'sensitivity\':results[\'sensitivity\'],\'models\':[fit[\'converged\'],coxfit[\'converged\']],\'checks\':results[\'checks\']},ensure_ascii=True),flush=True)\n', 'stock_analysis.py': "from pathlib import Path\nimport json,csv,zipfile,io,collections,re,datetime,unicodedata,statistics\nR=Path(__file__).resolve().parents[1];O=R/'data/complete';cut='2026-10-02'\ndef save(n,x):(O/n).write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding='utf8')\ndef fold(s):return ''.join(x for x in unicodedata.normalize('NFKD',s) if not unicodedata.combining(x)).upper()\ndef proc(s):\n m=re.fullmatch(r'(\\d{1,6})/(\\d{4})',str(s).strip().replace('.',''))\n if not m:return ''\n n=m[1].zfill(6);return n[:3]+'.'+n[3:]+'/'+m[2]\ndef csvwrite(n,rows):\n with (O/n).open('w',encoding='utf-8-sig',newline='') as f:\n  w=csv.DictWriter(f,fieldnames=list(rows[0]),delimiter=';');w.writeheader();w.writerows(rows)\ncfg=json.loads((O/'mineral_config.json').read_text(encoding='utf8'))\ndef minerals(s):\n f=fold(s);out={k for k,terms in cfg['tokens'].items() if any(re.search(r'\\b'+t+r'\\b',f) for t in terms)}\n if 'COLUMBITA' in f:out.add('nióbio')\n return sorted(out)\nsource=list(csv.DictReader((R/'data/raw/EstoqueAreas.csv').open(encoding='utf-8-sig'),delimiter=';'));by={};bad=[]\nfor x in source:\n p=proc(x['ProcessoMinerario'])\n if p:\n  if p in by:\n   # Preserve the first nonempty value; never overwrite fuller metadata with blanks.\n   by[p]={k:by[p].get(k) or x.get(k,'') for k in x}\n  else:by[p]=x\n else:bad.append(x['ProcessoMinerario'])\nz=zipfile.ZipFile(R/'data/raw/scm.zip')\ndef rows(n):\n path=next(x for x in z.namelist() if x.replace('\\\\','/').split('/')[-1]==n+'.txt')\n with io.TextIOWrapper(z.open(path),encoding='cp1252',errors='replace',newline='') as f:yield from csv.DictReader(f,delimiter=';')\nnames={x['IDEvento']:x['DSEvento'] for x in rows('Evento')};phases={x['IDFaseProcesso']:x['DSFaseProcesso'] for x in rows('FaseProcesso')}\nrules={\n 'entrada_disponibilidade':{'328','329','2463','2464','2865'},\n 'apta':{'2275'},\n 'selecionada':{'303','309','314','1123','1315','2465','2649','2651','2652'},\n 'livre':{'99','2466','2470','2471','2472','2473','2743'},\n 'suspensa':{'1313','2650'},\n 'revisao':{'357','1348','1349','1350','1351','1352','1353','1354','1355','1666','1667','1809','1827','2653','2777','2956'},\n 'oferta':{'305','310','677','1814','1836','2304','2305','2310','2338','2339','2467','2468','2469','2677','2802','2836'},\n 'recurso_pendente':{'360','1805','1887'},\n 'recurso_decidido':{'369','386','1806','1807','1888','1889','2963'},\n 'desbloqueio':{'1883','2883'},\n}\nstage={code:k for k,cs in rules.items() for code in cs};events=collections.defaultdict(list);invalid_dates=0;total=0\nfor x in rows('ProcessoEvento'):\n total+=1\n if x['IDEvento'] not in stage:continue\n t=x['DTEvento'][:10];p=proc(x['DSProcesso'])\n if not p or not re.fullmatch(r'\\d{4}-\\d{2}-\\d{2}',t) or t>'2026-10-02':continue\n try:datetime.date.fromisoformat(t)\n except ValueError:invalid_dates+=1;continue\n events[p].append((t,x['IDEvento']))\nprint('Stock relevant events',sum(map(len,events.values())),len(events),flush=True)\nP={};census=collections.Counter()\nfor x in rows('Processo'):\n p=proc(x['DSProcesso']);census[(x['IDFaseProcesso'],x['BTAtivo'])]+=1\n if p in by or p in events or x['IDFaseProcesso'] in ['8','15']:P[p]=x\nstatecodes=set().union(*(rules[k] for k in ['entrada_disponibilidade','apta','selecionada','livre','suspensa','revisao','oferta','desbloqueio']))\nentrances=rules['entrada_disponibilidade']|rules['apta'];qual=[];history=collections.Counter();flow=collections.Counter();cross=collections.Counter();overlap=collections.Counter();yearly=collections.defaultdict(collections.Counter);eventcount=collections.Counter()\nfor p,x in P.items():\n ev=sorted(set(events.get(p,[])));oper=[e for e in ev if e[1] in statecodes];lastdate=oper[-1][0] if oper else '';same=[e for e in oper if e[0]==lastdate];states={stage[e[1]] for e in same};latest=next(iter(states)) if len(states)==1 else 'ambigua_mesma_data' if states else 'sem_evento_estado'\n entering=[e for e in oper if e[1] in entrances];lastentry=entering[-1][0] if entering else ''\n age=(datetime.date.fromisoformat(cut)-datetime.date.fromisoformat(lastentry)).days if lastentry else None\n rec=[e for e in ev if e[1] in rules['recurso_pendente']];dec=[e for e in ev if e[1] in rules['recurso_decidido']];appeal=bool(rec and (not dec or rec[-1][0]>dec[-1][0]))\n sx=by.get(p);status=sx['SituacaoEstoque'] if sx else 'Fora do arquivo SOPLE';active=x['BTAtivo']=='S';phase=x['IDFaseProcesso'];mins=minerals(sx['Substancia']) if sx else []\n discrepancy=bool(sx and status=='Apta para Disponibilidade' and ((not active) or latest in ['selecionada','livre','suspensa','revisao','ambigua_mesma_data'] or appeal))\n overlap['scm_active_8_15' if active and phase in ['8','15'] else 'other']+=1\n if sx:cross[(status,phase,x['BTAtivo'])]+=1\n for t,c in ev:eventcount[c]+=1\n for t,c in oper:flow[(t[:4],stage[c])]+=1\n for year in range(2020,2027):\n  limit=f'{year}-12-31' if year<2026 else cut;relevant=[e for e in oper if e[0]<=limit]\n  if relevant:\n   lt=relevant[-1][0];ss={stage[c] for t,c in relevant if t==lt};yearly[str(year)][next(iter(ss)) if len(ss)==1 else 'ambigua_mesma_data']+=1\n qual.append({'process':p,'active':active,'phase':phase,'phase_name':phases.get(phase,''),'sople':status,'state':sx['UnidadeFederacao'] if sx else '', 'city':sx['Municipio'] if sx else '', 'substance':sx['Substancia'] if sx else '', 'minerals':' | '.join(mins),'proxy':bool(mins),'nominated':sx['IndicaNominacao'] if sx else '', 'listed':sx['IndicaDisponibilizacaoEdital'] if sx else '', 'last_operational_event':lastdate,'last_state':latest,'last_codes':' | '.join(c for t,c in same),'last_entry':lastentry,'age_days':age,'appeal_unresolved_signal':appeal,'apt_conflict':discrepancy,'entry_records':len(entering),'reentry':len(entering)>1})\ncsvwrite('stock_process_audit.csv',qual)\nactive=[r for r in qual if r['active'] and r['phase'] in ['8','15']];sople=[r for r in qual if r['sople']!='Fora do arquivo SOPLE'];apt=[r for r in sople if r['sople']=='Apta para Disponibilidade'];opencandidates=[r for r in qual if r['last_state'] in ['entrada_disponibilidade','apta','oferta','desbloqueio']]\ndef age_summary(rs):\n ages=[r['age_days'] for r in rs if r['age_days'] is not None]\n return {'n':len(rs),'dated':len(ages),'without_entry':len(rs)-len(ages),'median_days':statistics.median(ages) if ages else None,'over2years':sum(a>730 for a in ages),'over5years':sum(a>1826 for a in ages),'over10years':sum(a>3652 for a in ages),'unresolved_appeal':sum(r['appeal_unresolved_signal'] for r in rs),'proxy':sum(r['proxy'] for r in rs),'conflict':sum(r['apt_conflict'] for r in rs),'reentry':sum(r['reentry'] for r in rs)}\nresult={'date':cut,'raw_rows':len(source),'valid_sople_keys':len(by),'invalid_identifiers':len(bad),'invalid_examples':bad[:10],'scm_processes_examined':sum(census.values()),'scm_event_rows_examined':total,'event_coverage':len(events),'active_phase_stock':age_summary(active),'sople_apt':age_summary(apt),'event_open_candidates':age_summary(opencandidates),'sople_status':{k:age_summary([r for r in sople if r['sople']==k]) for k in sorted({r['sople'] for r in sople})},'phase_census':[{'phase':p,'active':a,'n':n} for (p,a),n in census.items()],'crosswalk':[{'sople':s,'phase':p,'active':a,'n':n} for (s,p,a),n in cross.items()],'active_in_sople':sum(r['sople']!='Fora do arquivo SOPLE' for r in active),'active_missing_sople':sum(r['sople']=='Fora do arquivo SOPLE' for r in active),'apt_active_8_15':sum(r['active'] and r['phase'] in ['8','15'] for r in apt),'apt_current_states':dict(collections.Counter(r['last_state'] for r in apt)),'yearly_event_states':{k:dict(v) for k,v in yearly.items()},'annual_event_counts':[{'year':y,'state':s,'n':n} for (y,s),n in flow.items() if y>='2020'],'regions':{uf:age_summary([r for r in sople if r['state']==uf]) for uf in sorted({r['state'] for r in sople})},'strategic_stock':{m:age_summary([r for r in sople if m in r['minerals'].split(' | ')]) for m in cfg['tokens']},'invalid_dates':invalid_dates,'event_rules':[{'code':c,'label':names[c],'state':stage[c],'records':eventcount[c],'justification':{'entrada_disponibilidade':'Entrada ou retorno explícito; exige leitura jurídica do caso.','apta':'Aptidão cadastral registrada; conferir requisitos atuais.','selecionada':'Seleção registrada, não comprovação de pagamento ou outorga.','livre':'Liberação explícita registrada; não converter em oferta disponível.','suspensa':'Suspensão ou bloqueio explícito.','revisao':'Ato cancelado, excluído ou sem efeito; não presumir retorno automático.','oferta':'Inclusão em procedimento, sem decisão final presumida.','recurso_pendente':'Sinal de recurso; protocolo não determina sozinho efeito suspensivo.','recurso_decidido':'Registro de decisão ou desistência; conferir recurso atingido.','desbloqueio':'Registro de desbloqueio, sem certificação integral de aptidão.'}[stage[c]]} for c in sorted(stage,key=int)],'interpretation':'Estoque por eventos é proxy histórica de estados documentados; não certifica disponibilidade jurídica nem aplica automaticamente prazo legal de dois anos.'}\nsave('stock_results.json',result);csvwrite('stock_event_codebook.csv',result['event_rules']);save('stock_conflicts.json',[r for r in apt if r['apt_conflict']])\nprint(json.dumps({k:result[k] for k in ['raw_rows','valid_sople_keys','invalid_identifiers','active_phase_stock','sople_apt','event_open_candidates','active_missing_sople']},ensure_ascii=True),flush=True)\n", 'refinements_pp.py': '"""Five explicit improvement passes on the computed analysis."""\nfrom pathlib import Path\nimport json,csv,collections,math,re,unicodedata\nimport numpy as np\nfrom statistics_pp import logistic,incidence\nR=Path(__file__).resolve().parents[1];O=R/\'data/complete\'\nA=json.loads((O/\'results.json\').read_text(encoding=\'utf8\'));B=json.loads((O/\'stock_results.json\').read_text(encoding=\'utf8\'));C=json.loads((O/\'mineral_config.json\').read_text(encoding=\'utf8\'))\nraw=list(csv.DictReader((O/\'award_analysis.csv\').open(encoding=\'utf-8-sig\'),delimiter=\';\'))\nrows=[]\nfor r in raw:\n for k in [\'round\',\'strategic\',\'cession\',\'unpaid\',\'request\',\'approval\',\'denial\',\'holder\',\'holder_n\',\'cessions\',\'corporate\',\'cause\',\'time\',\'followup\',\'payment_record\']:r[k]=int(r[k])\n for k in [\'bid\',\'ha\',\'cfem\']:r[k]=float(r[k])\n for k in [\'first_days\',\'exit_days\']:r[k]=int(r[k]) if r[k] else None\n r[\'fiscal_eligible\']=r[\'fiscal_eligible\']==\'True\';rows.append(r)\ndef fold(s):return \'\'.join(x for x in unicodedata.normalize(\'NFKD\',s) if not unicodedata.combining(x)).upper()\ndef summary(rs):return {\'n\':len(rs),\'bid\':sum(r[\'bid\'] for r in rs),\'unpaid\':sum(r[\'unpaid\'] for r in rs),\'unpaid_value\':sum(r[\'bid\'] for r in rs if r[\'unpaid\']),\'cession\':sum(r[\'cession\'] for r in rs),\'holder\':sum(r[\'holder\'] for r in rs),\'cfem\':sum(r[\'cfem\'] for r in rs if r[\'fiscal_eligible\']),\'cfem_positive\':sum(r[\'cfem\']>0 and r[\'fiscal_eligible\'] for r in rs)}\nreport=[]\nreport.append({\'pass\':1,\'question\':\'A mesma ocorrência pode ser atribuída a dois ciclos sucessivos?\',\'changes\':\'Encerrados os ramos quando o próprio associado participa de nova arrematação. Excluídos acontecimentos posteriores do ciclo anterior.\',\'evidence\':{\'occurrences_before\':1112,\'occurrences_after\':A[\'overall\'][\'cessions\'],\'unpaid_current\':A[\'sensitivity\'][\'unpaid_current\'],\'unpaid_preserved\':A[\'overall\'][\'unpaid\']},\'check\':\'A nova regra removeu uma ocorrência e conciliou os 1.272 sinais do retrato atualizado com a base preservada.\'})\n# Pass 2: distinguish direct names from inferred mineral carriers and missing substance.\nunknown_terms={\'\',\'NAO SE APLICA\',\'NAO INFORMADA\',\'NAO INFORMADO\'}\nfor r in rows:\n f=fold(r[\'substances\']);r[\'known\']=f not in unknown_terms\n r[\'direct_minerals\']=[m for m in C[\'tokens\'] if re.search(r\'\\b\'+fold(m)+r\'\\b\',f)]\n r[\'direct_proxy\']=bool(r[\'direct_minerals\'])\n r[\'group_I\']=any(m in C[\'groups\'][\'I\'] for m in r[\'minerals\'].split(\' | \'));r[\'group_II\']=any(m in C[\'groups\'][\'II\'] for m in r[\'minerals\'].split(\' | \'));r[\'group_III\']=any(m in C[\'groups\'][\'III\'] for m in r[\'minerals\'].split(\' | \'))\nA[\'mineral_classification_sensitivity\']={\'broad_proxy\':summary([r for r in rows if r[\'strategic\']]),\'direct_names_only\':summary([r for r in rows if r[\'direct_proxy\']]),\'other_known\':summary([r for r in rows if r[\'known\'] and not r[\'strategic\']]),\'unknown\':summary([r for r in rows if not r[\'known\']])}\nA[\'policy_groups\']={k:summary([r for r in rows if r[\'group_\'+k]]) for k in [\'I\',\'II\',\'III\']}\nA[\'strategic_incidence\']=incidence([r[\'time\'] for r in rows if r[\'strategic\']],[r[\'cause\'] for r in rows if r[\'strategic\']])\nA[\'other_known_incidence\']=incidence([r[\'time\'] for r in rows if not r[\'strategic\'] and r[\'known\']],[r[\'cause\'] for r in rows if not r[\'strategic\'] and r[\'known\']])\nA[\'critical_territories\']={s:summary([r for r in rows if r[\'state\']==s and r[\'strategic\']]) for s in sorted({r[\'state\'] for r in rows if r[\'strategic\']})}\nA[\'critical_round_matrix\']={m:{str(k):sum(m in r[\'minerals\'].split(\' | \') and r[\'round\']==k for r in rows) for k in [1,2,3,4,5,8]} for m in C[\'tokens\']}\nreport.append({\'pass\':2,\'question\':\'O recorte mineral depende de sinônimos ou de informação ausente?\',\'changes\':\'Executada classificação restrita por nomes diretos, ampla por portadores explicitados e separação dos não informados. Grupos I, II e III tratados com sobreposição.\',\'evidence\':A[\'mineral_classification_sensitivity\'],\'check\':\'A lista de 2021 é proxy histórica. Não foi rebatizada como lista legal de minerais críticos da PNMCE.\'})\n# Pass 3: additional regional controls and missing-substance sensitivity.\nregions={\'Norte\':{\'Acre\',\'Amapá\',\'Amazonas\',\'Pará\',\'Rondônia\',\'Roraima\',\'Tocantins\'},\'Nordeste\':{\'Alagoas\',\'Bahia\',\'Ceará\',\'Maranhão\',\'Paraíba\',\'Pernambuco\',\'Piauí\',\'Rio Grande do Norte\',\'Sergipe\'},\'Centro-Oeste\':{\'Distrito Federal\',\'Goiás\',\'Mato Grosso\',\'Mato Grosso do Sul\'},\'Sul\':{\'Paraná\',\'Rio Grande do Sul\',\'Santa Catarina\'}}\nmr=[r for r in rows if r[\'round\']!=1 and r[\'ha\']>0 and r[\'known\']]\nnames=[\'Intercepto\',\'Log do lance\',\'Log da área\',\'Lavra\',\'Proxy estratégica\',\'Rodada 3\',\'Rodada 4\',\'Rodada 5\',\'Rodada 8\']+list(regions)\nX=np.array([[1,math.log1p(r[\'bid\']),math.log1p(r[\'ha\']),int(\'Lavra\' in r[\'regime\']),r[\'strategic\'],*[r[\'round\']==k for k in [3,4,5,8]],*[r[\'state\'] in s for s in regions.values()]] for r in mr],float)\nfit=logistic(X,[r[\'unpaid\'] for r in mr],[r[\'origin\'] for r in mr]);fit.pop(\'probabilities\');fit[\'variables\']=names;A[\'logistic_region_known\']=fit\n# Fixed horizon first cession as robustness to proportional hazard assumptions.\nfr=[r for r in mr if r[\'followup\']>=730];fn=[\'Intercepto\',\'Log do lance\',\'Log da área\',\'Lavra\',\'Proxy estratégica\',\'Rodada 3\',\'Rodada 4\',\'Rodada 5\']\nFX=np.array([[1,math.log1p(r[\'bid\']),math.log1p(r[\'ha\']),int(\'Lavra\' in r[\'regime\']),r[\'strategic\'],*[r[\'round\']==k for k in [3,4,5]]] for r in fr],float);ff=logistic(FX,[r[\'first_days\'] is not None and r[\'first_days\']<=730 for r in fr],[r[\'origin\'] for r in fr]);ff.pop(\'probabilities\');ff[\'variables\']=fn;A[\'logistic_cession730\']=ff\nreport.append({\'pass\':3,\'question\':\'As associações resistem à maturidade, à geografia e à falta de substância?\',\'changes\':\'Estimado logit com macrorregião e somente substâncias conhecidas. Estimado logit de cessão em 730 dias com seguimento completo. Comparado ao Cox e à exclusão do 1% de maiores lances.\',\'evidence\':{\'nonpayment_n\':fit[\'n\'],\'nonpayment_converged\':fit[\'converged\'],\'cession730_n\':ff[\'n\'],\'cession730_converged\':ff[\'converged\']},\'check\':\'Modelos convergentes. Coeficientes descrevem associações observacionais; não são efeitos causais de preço mínimo.\'})\n# Pass 4: stock quality quarantine, active/event proxy, matrix and age distributions.\nstock=list(csv.DictReader((O/\'stock_process_audit.csv\').open(encoding=\'utf-8-sig\'),delimiter=\';\'))\nfor r in stock:\n for k in [\'active\',\'proxy\',\'appeal_unresolved_signal\',\'apt_conflict\']:r[k]=r[k]==\'True\'\n r[\'age_days\']=int(r[\'age_days\']) if r[\'age_days\'] else None\nvalid=[r for r in stock if r[\'sople\']!=\'Fora do arquivo SOPLE\'];openactive=[r for r in stock if r[\'active\'] and r[\'last_state\'] in [\'entrada_disponibilidade\',\'apta\',\'oferta\',\'desbloqueio\']]\nB[\'event_open_active_only\']={\'n\':len(openactive),\'in_sople\':sum(r[\'sople\']!=\'Fora do arquivo SOPLE\' for r in openactive),\'phase_8_15\':sum(r[\'phase\'] in [\'8\',\'15\'] for r in openactive)}\nB[\'proxy_by_status\']={k:{\'total\':len(rs:=[r for r in valid if r[\'sople\']==k]),\'proxy\':sum(r[\'proxy\'] for r in rs),\'without_substance\':sum(fold(r[\'substance\']) in unknown_terms for r in rs),\'appeal_signal\':sum(r[\'appeal_unresolved_signal\'] for r in rs),\'older2\':sum(r[\'age_days\'] is not None and r[\'age_days\']>730 for r in rs)} for k in sorted({r[\'sople\'] for r in valid})}\nB[\'strategic_apt_minerals\']={m:{\'apt\':len(rs:=[r for r in valid if r[\'sople\']==\'Apta para Disponibilidade\' and m in r[\'minerals\'].split(\' | \')]),\'older2\':sum(r[\'age_days\'] is not None and r[\'age_days\']>730 for r in rs),\'conflict\':sum(r[\'apt_conflict\'] for r in rs),\'median_days\':float(np.median(a)) if (a:=[r[\'age_days\'] for r in rs if r[\'age_days\'] is not None]) else None} for m in C[\'tokens\']}\nB[\'age_distribution\']={group:{label:sum((r[\'age_days\'] is not None and lo<=r[\'age_days\']<hi) for r in rs) for label,lo,hi in [(\'Até 1 ano\',0,366),(\'1 a 2 anos\',366,731),(\'2 a 5 anos\',731,1827),(\'5 a 10 anos\',1827,3653),(\'Mais de 10 anos\',3653,100000)]} for group,rs in [(\'SCM ativo fases 8 e 15\',[r for r in stock if r[\'active\'] and r[\'phase\'] in [\'8\',\'15\']]),(\'Aptas SOPLE\',[r for r in valid if r[\'sople\']==\'Apta para Disponibilidade\'])]}\nreport.append({\'pass\':4,\'question\':\'O estoque incorpora registros inválidos ou históricos como se fossem áreas ofertáveis?\',\'changes\':\'Quarentena dos 82 identificadores inválidos, normalização de chaves e comparação de estoque ativo, aptidão SOPLE e estados por evento. Filtro de atividade aplicado também à proxy histórica.\',\'evidence\':{\'raw_rows\':B[\'raw_rows\'],\'valid_keys\':B[\'valid_sople_keys\'],\'active_stock\':B[\'active_phase_stock\'][\'n\'],\'apt_sople\':B[\'sople_apt\'][\'n\'],\'apt_conflicts\':B[\'sople_apt\'][\'conflict\'],\'event_open_active\':B[\'event_open_active_only\']},\'check\':\'Nenhum texto inválido de identificador foi executado ou convertido à força em processo. Idade não foi interpretada como liberação automática.\'})\n# Pass 5: institutional decision thresholds and internal consistency.\nbykey={r[\'key\']:r for r in rows};assert len(bykey)==7528;assert abs(sum(r[\'bid\'] for r in rows)-907604126.29)<.01\nassert sum(r[\'unpaid\'] for r in rows)==1272\nassert all(r[\'first_days\'] is None or 0<=r[\'first_days\']<=r[\'followup\'] for r in rows)\nfor v in A[\'incidence\'][\'curve\']:assert abs(v[\'incidence\']+v[\'exit_incidence\']+v[\'survival\']-1)<1e-9\nfor fitname in [\'logistic_nonpayment\',\'logistic_without_top1\',\'cox_cession\',\'logistic_region_known\',\'logistic_cession730\']:assert A[fitname][\'converged\']\nA[\'fiscal_unique\']={\'origins_eligible\':len({r[\'origin\'] for r in rows if r[\'fiscal_eligible\']}),\'positive\':len({r[\'origin\'] for r in rows if r[\'fiscal_eligible\'] and r[\'cfem\']>0})}\nreport.append({\'pass\':5,\'question\':\'O documento permite refazer os resultados e usar as conclusões sem confundir cenário com evidência?\',\'changes\':\'Executadas identidades dos totais financeiros, unicidade dos ciclos, limites temporais, soma das probabilidades e convergência dos modelos. Cenários de incentivos rotulados como hipóteses, separados da triagem histórica.\',\'evidence\':{\'cycles\':len(bykey),\'total_bid\':sum(r[\'bid\'] for r in rows),\'all_models_converged\':True,\'fiscal_unique\':A[\'fiscal_unique\']},\'check\':\'Resultados, parâmetros, tabelas por processo e dicionário de eventos exportados para reprodução. Não há preço ótimo causal alegado.\'})\nfor n,x in [(\'results.json\',A),(\'stock_results.json\',B),(\'five_review_passes.json\',report)]:\n (O/n).write_text(json.dumps(x,ensure_ascii=False,indent=2,default=lambda v:v.item() if isinstance(v,np.generic) else str(v)),encoding=\'utf8\')\nprint(json.dumps({\'five_passes\':len(report),\'classification\':A[\'mineral_classification_sensitivity\'],\'regional_model_or\':dict(zip(names,fit[\'or\'])),\'stock_active_events\':B[\'event_open_active_only\'],\'fiscal\':A[\'fiscal_unique\']},ensure_ascii=True))\n'}
for name, content in SCRIPTS.items():
    (ROOT/"src"/name).write_text(content, encoding="utf-8")
BASELINE = '...'
(ROOT/"data/raw/analitico_preservado.json").write_bytes(gzip.decompress(base64.b64decode(BASELINE)))
print("Scripts e base mínima preservada gravados.")


## 1. Insumos oficiais

Para reprodução exata, envie os cinco arquivos do extrato preservado ao diretório data/raw antes de executar a próxima célula. Arquivos ausentes serão baixados. Se o hash diferir, a execução será interrompida por padrão. A alteração explícita de PERMITIR_EXTRATO_DIFERENTE autoriza uma atualização, que não reproduz exatamente os números publicados.

In [ ]:
MANIFEST = [{'file': 'scm.zip', 'url': 'https://dadosabertos.anm.gov.br/SCM/microdados/microdados-scm.zip', 'bytes': 181010122, 'sha256': 'fde645bd8f21b6e0faccb04993688b9efd90d988d97d921b319a5c97175c9cee'}, {'file': 'Cessoes_de_Direitos.csv', 'url': 'https://dadosabertos.anm.gov.br/SCM/Cessoes_de_Direitos.csv', 'bytes': 13650327, 'sha256': 'd35f7d4762173d8fb6103d50a4d0c2bc712832b7eb45a768c91679571b2b6635'}, {'file': 'ResultadoRodadaDisponibilidade.csv', 'url': 'https://dadosabertos.anm.gov.br/SOPLE/ResultadoRodadaDisponibilidade.csv', 'bytes': 3646309, 'sha256': '25f5eca7ba5ecf038a7de4903d30f398cd0da7d916679e58cd6d31f121fd5ff2'}, {'file': 'ResultadoAvaliacaoSocial.csv', 'url': 'https://dadosabertos.anm.gov.br/SOPLE/ResultadoAvaliacaoSocial.csv', 'bytes': 60155, 'sha256': '87768b9ea7dee154e037c9e0958c2bce25084c6d5dc9e33e21f55c602f9c1f1f'}, {'file': 'EstoqueAreas.csv', 'url': 'https://dadosabertos.anm.gov.br/SOPLE/EstoqueAreas.csv', 'bytes': 21783580, 'sha256': 'a26fb1aeb34357a35012b3a488df5bc5e483a7720695086b1e73a5fc3f1ed3e4'}]
PERMITIR_EXTRATO_DIFERENTE = False
for item in MANIFEST:
    dest = ROOT/"data/raw"/item["file"]
    if not dest.exists():
        request = urllib.request.Request(item["url"], headers={"User-Agent":"Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=180) as response, dest.open("wb") as out:
            import shutil
            shutil.copyfileobj(response, out)
    actual = hashlib.sha256(dest.read_bytes()).hexdigest()
    same = actual == item["sha256"]
    print(item["file"], "extrato idêntico" if same else "EXTRATO DIFERENTE")
    if not same and not PERMITIR_EXTRATO_DIFERENTE:
        raise RuntimeError("O insumo mudou. Use o extrato preservado ou autorize explicitamente uma nova atualização.")


## 2. Reconstrução e estimação

A leitura de milhões de eventos exige memória e tempo. A ordem abaixo é necessária: extrair o cadastro, reconstruir ciclos e modelos, apurar o estoque nacional e executar as cinco revisões. Nenhuma operação modifica o artigo V12 ou o painel anterior.

In [ ]:
for script in ["extract_scm.py", "complete_analysis.py", "stock_analysis.py", "refinements_pp.py"]:
    print("Executando", script, flush=True)
    subprocess.run([sys.executable, str(ROOT/"src"/script)], cwd=ROOT, check=True)


## 3. Conferência dos resultados

A atualização pode produzir totais diferentes. As verificações de referência pertencem ao extrato preservado. Os arquivos CSV usam ponto e vírgula e UTF-8 com BOM para leitura no Excel.

In [ ]:
A = json.loads((ROOT/"data/complete/results.json").read_text(encoding="utf8"))
B = json.loads((ROOT/"data/complete/stock_results.json").read_text(encoding="utf8"))
print(json.dumps({"cessoes":A["overall"],"estoque_ativo":B["active_phase_stock"],"aptas":B["sople_apt"]},ensure_ascii=False,indent=2))


## 4. Análise crítica

Verifique a classificação mineral, os 82 identificadores inválidos em quarentena, a diferença entre fase e aptidão e a cobertura fiscal. Cessão não prova especulação. Idade cadastral não prova infração. Complemento do sinal de não pagamento não é receita confirmada. Não há preço mínimo ótimo identificado a partir apenas dos vencedores.

## 5. Ampliação territorial e estatística de 03/10/2026

Esta seção reproduz os cruzamentos e diagnósticos da revisão 4. Contém insumos analíticos sem nomes de vencedores ou identificadores fiscais pessoais. As geometrias originais devem corresponder aos hashes preservados. As rotinas foram executadas localmente; este caderno não declara execução na nuvem. A revisão humana e o piloto de campo não foram realizados.

Instale as dependências antes de executar. O processamento espacial usa memória substancial. A falha de uma fonte ou uma mudança de hash deve ser investigada, não ignorada.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'statsmodels', 'shapely>=2.1', 'pyproj', 'pyshp'], check=True)


In [ ]:
EXPANSION_SCRIPTS = {'statistics_pp.py': '"""Numpy implementations with explicit assumptions and deterministic checks."""\nimport numpy as np,math\n\ndef logistic(X,y,clusters=None,max_iter=150):\n X=np.asarray(X,float);y=np.asarray(y,float);b=np.zeros(X.shape[1]);ll=-np.inf\n for it in range(max_iter):\n  eta=np.clip(X@b,-35,35);p=1/(1+np.exp(-eta));w=np.maximum(p*(1-p),1e-9)\n  H=X.T@(X*w[:,None]);score=X.T@(y-p);step=np.linalg.solve(H+np.eye(len(b))*1e-9,score)\n  old=np.sum(y*eta-np.logaddexp(0,eta));scale=1\n  while scale>1e-7:\n   candidate=b+scale*step;v=X@candidate;new=np.sum(y*v-np.logaddexp(0,v))\n   if new>=old-1e-8:break\n   scale/=2\n  b=candidate\n  if np.max(np.abs(scale*step))<1e-8:break\n eta=X@b;p=1/(1+np.exp(-np.clip(eta,-35,35)));H=X.T@(X*(p*(1-p))[:,None]);bread=np.linalg.pinv(H)\n u=X*(y-p)[:,None]\n if clusters is not None:\n  keys,inv=np.unique(clusters,return_inverse=True);scores=np.zeros((len(keys),len(b)));np.add.at(scores,inv,u)\n  correction=len(keys)/max(1,len(keys)-1)*(len(y)-1)/(len(y)-len(b));cov=bread@(scores.T@scores)@bread*correction\n else:cov=bread@(u.T@u)@bread*len(y)/(len(y)-len(b))\n se=np.sqrt(np.maximum(np.diag(cov),0));z=np.divide(b,se,out=np.zeros_like(b),where=se>0)\n return {\'coef\':b.tolist(),\'se\':se.tolist(),\'or\':np.exp(b).tolist(),\'lo\':np.exp(b-1.96*se).tolist(),\'hi\':np.exp(b+1.96*se).tolist(),\'p\':[math.erfc(abs(x)/math.sqrt(2)) for x in z],\'n\':len(y),\'events\':int(sum(y)),\'iterations\':it+1,\'converged\':bool(np.max(np.abs(scale*step))<1e-6),\'score_max\':float(np.max(np.abs(X.T@(y-p)))),\'rank\':int(np.linalg.matrix_rank(X)),\'condition\':float(np.linalg.cond(H)),\'brier\':float(np.mean((p-y)**2)),\'probabilities\':p.tolist()}\n\ndef incidence(times,causes,horizons=(180,365,730,1095,1460,1825)):\n """Aalen-Johansen competing first events. cause 0=censor, 1=target, 2=exit."""\n t=np.asarray(times,int);c=np.asarray(causes,int);S=1.;F=0.;F2=0.;rows=[]\n all_sorted=np.sort(t);event_times=np.unique(t[c>0]);u1,c1=np.unique(t[c==1],return_counts=True);u2,c2=np.unique(t[c==2],return_counts=True);n1=dict(zip(u1,c1));n2=dict(zip(u2,c2))\n for x in event_times:\n  risk=int(len(t)-np.searchsorted(all_sorted,x));d1=int(n1.get(x,0));d2=int(n2.get(x,0))\n  F+=S*d1/risk;F2+=S*d2/risk;S*=1-(d1+d2)/risk\n  rows.append({\'day\':int(x),\'risk\':risk,\'events\':d1,\'exits\':d2,\'incidence\':F,\'exit_incidence\':F2,\'survival\':S})\n horizon=[]\n for h in horizons:\n  available=h<=max(t,default=0);v=[r for r in rows if r[\'day\']<=h];last=v[-1] if v else {\'incidence\':0.,\'exit_incidence\':0.,\'survival\':1.}\n  horizon.append({\'day\':h,\'available\':available,\'at_risk\':int(sum(t>=h)),\'incidence\':last[\'incidence\'] if available else None,\'exit_incidence\':last[\'exit_incidence\'] if available else None,\'survival\':last[\'survival\'] if available else None})\n return {\'curve\':rows,\'horizons\':horizon}\n\ndef cox(X,t,e,max_iter=60):\n """Cause-specific proportional hazards; Breslow ties, model-based covariance."""\n X=np.asarray(X,float);t=np.asarray(t,int);e=np.asarray(e,int)\n order=np.argsort(-t,kind=\'stable\');X=X[order];t=t[order];e=e[order];b=np.zeros(X.shape[1]);unique=np.unique(t[e==1]);idx=np.array([np.where(t>=v)[0][-1] for v in unique]);d=np.array([sum((t==v)&(e==1)) for v in unique]);xev=np.array([X[(t==v)&(e==1)].sum(0) for v in unique]);\n def calc(beta):\n  eta=X@beta;w=np.exp(np.clip(eta,-30,30));s0=np.cumsum(w)[idx];s1=np.cumsum(w[:,None]*X,axis=0)[idx];s2=np.cumsum(w[:,None,None]*X[:,:,None]*X[:,None,:],axis=0)[idx];m=s1/s0[:,None]\n  ll=float((X[e==1]@beta).sum()-np.sum(d*np.log(s0)));g=(xev-d[:,None]*m).sum(0);H=np.sum(d[:,None,None]*(s2/s0[:,None,None]-m[:,:,None]*m[:,None,:]),axis=0)\n  return ll,g,H\n for it in range(max_iter):\n  ll,g,H=calc(b);step=np.linalg.solve(H+np.eye(len(b))*1e-9,g);scale=1\n  while scale>1e-7 and calc(b+scale*step)[0]<ll-1e-9:scale/=2\n  b+=scale*step\n  if np.max(abs(scale*step))<1e-8:break\n ll,g,H=calc(b);se=np.sqrt(np.maximum(0,np.diag(np.linalg.pinv(H))))\n return {\'coef\':b.tolist(),\'se\':se.tolist(),\'hr\':np.exp(b).tolist(),\'lo\':np.exp(b-1.96*se).tolist(),\'hi\':np.exp(b+1.96*se).tolist(),\'n\':len(t),\'events\':int(sum(e)),\'converged\':bool(np.max(abs(scale*step))<1e-6),\'score_max\':float(max(abs(g))),\'iterations\':it+1,\'covariance\':\'Model-based inverse information; observational association, Breslow ties.\'}\n\ndef check():\n # Known intercept-only MLE.\n fit=logistic(np.ones((100,1)),np.r_[np.ones(30),np.zeros(70)])\n assert abs(fit[\'coef\'][0]-math.log(.3/.7))<1e-6\n # First event and competing exit among four units: each cumulative incidence is .25.\n a=incidence([1,2,3,4],[1,2,0,0],horizons=[2])[\'horizons\'][0]\n assert abs(a[\'incidence\']-.25)<1e-9 and abs(a[\'exit_incidence\']-.25)<1e-9\n assert abs(a[\'survival\']-.5)<1e-9\n # Numerical gradient/Hessian validation for Cox by a simulated known direction.\n rng=np.random.default_rng(721);x=rng.normal(size=(600,1));t=np.round(rng.exponential(100/np.exp(.7*x[:,0]))).astype(int)+1;c=np.full(600,200);m=cox(x,np.minimum(t,c),(t<=c).astype(int));assert .35<m[\'coef\'][0]<1.1 and m[\'converged\']\n return {\'logistic_intercept\':\'passed\',\'competing_risk_identity\':\'passed\',\'cox_simulated_direction\':\'passed\'}\n\nif __name__==\'__main__\':print(check())\n', 'territorial_expansion.py': '"""Exact portfolio geometry operations; simplified geometries are display-only."""\nfrom pathlib import Path\nimport sys,json,csv,zipfile,io,re,hashlib,collections,unicodedata,logging\nimport numpy as np\nR=Path(__file__).resolve().parents[1];sys.path.insert(0,str(R.parent/\'tmp/geo-libs\'))\nimport shapefile,shapely\nfrom shapely.geometry import shape,mapping,Point\nfrom shapely.ops import transform,unary_union\nfrom shapely import STRtree,make_valid\nfrom pyproj import Transformer,Geod\nlogging.getLogger(\'shapefile\').setLevel(logging.ERROR)\nD=R/\'data/expansion\';RAW=D/\'raw\';D.mkdir(exist_ok=True)\ndef save(n,x):(D/n).write_text(json.dumps(x,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False),encoding=\'utf8\')\ndef fold(s):return \'\'.join(c for c in unicodedata.normalize(\'NFKD\',str(s)) if not unicodedata.combining(c)).upper()\ndef proc(s):\n m=re.fullmatch(r\'(\\d{1,6})/(\\d{4})\',str(s).replace(\'.\',\'\').strip())\n return m[1].zfill(6)[:3]+\'.\'+m[1].zfill(6)[3:]+\'/\'+m[2] if m else \'\'\nrows=list(csv.DictReader((R/\'data/complete/stock_process_audit.csv\').open(encoding=\'utf-8-sig\'),delimiter=\';\'))\nstock={r[\'process\']:r for r in rows if r[\'sople\']==\'Apta para Disponibilidade\' or (r[\'active\']==\'True\' and r[\'phase\'] in [\'8\',\'15\'])}\nassert len(stock)==30861\ncfg=json.loads((R/\'data/complete/mineral_config.json\').read_bytes())\ndef minerals(s):\n f=fold(s);out={m for m,terms in cfg[\'tokens\'].items() if any(re.search(r\'\\b\'+fold(t)+r\'\\b\',f) for t in terms)}\n if \'COLUMBITA\' in f:out.add(\'nióbio\')\n return out\ndef reader(file,stem):\n z=zipfile.ZipFile(RAW/file)\n return z,shapefile.Reader(shp=io.BytesIO(z.read(stem+\'.shp\')),shx=io.BytesIO(z.read(stem+\'.shx\')),dbf=io.BytesIO(z.read(stem+\'.dbf\')),encoding=\'utf8\')\nforward=Transformer.from_crs(\'EPSG:4674\',\'+proj=laea +lat_0=-15 +lon_0=-54 +ellps=GRS80 +units=m +no_defs\',always_xy=True).transform\nreverse=Transformer.from_crs(\'+proj=laea +lat_0=-15 +lon_0=-54 +ellps=GRS80 +units=m +no_defs\',\'EPSG:4674\',always_xy=True).transform\ngeod=Geod(ellps=\'GRS80\')\ndef polygon(g):\n if not g.is_valid:g=make_valid(g)\n if g.geom_type in [\'Polygon\',\'MultiPolygon\']:return g\n if hasattr(g,\'geoms\'):return unary_union([p for p in g.geoms if p.geom_type in [\'Polygon\',\'MultiPolygon\']])\n return None\nz,s=reader(\'SIGMINE_BRASIL.zip\',\'BRASIL\');seen=collections.defaultdict(list);metadata={};invalid=0;empty=0\nfor i,rec in enumerate(s.iterRecords(fields=[\'PROCESSO\',\'AREA_HA\',\'FASE\',\'SUBS\',\'UF\'])):\n r=rec.as_dict();key=proc(r[\'PROCESSO\'])\n if key not in stock:continue\n g=shape(s.shape(i).__geo_interface__)\n if not g.is_valid:invalid+=1\n g=polygon(g)\n if g is None or g.is_empty:empty+=1;continue\n pg=polygon(transform(forward,g))\n if pg is None or pg.is_empty:empty+=1;continue\n seen[key].append(pg);metadata[key]={\'sigmine_phase\':r[\'FASE\'],\'sigmine_substance\':r[\'SUBS\'],\'sigmine_uf\':r[\'UF\'],\'area_declared_ha\':float(r[\'AREA_HA\'] or 0)}\ngeoms={k:unary_union(gs) for k,gs in seen.items()};print(\'matched\',len(geoms),\'invalid repaired\',invalid,flush=True)\nzb,b=reader(\'BLOQUEIO.zip\',\'BLOQUEIO\');blocks=[];blockmeta=[]\nfor sr in b.iterShapeRecords():\n g=polygon(shape(sr.shape.__geo_interface__))\n if g is None or g.is_empty:continue\n blocks.append(polygon(transform(forward,g)));blockmeta.append({k:str(v) for k,v in sr.record.as_dict().items()})\nbt=STRtree(blocks);geo_context=json.loads((RAW/\'sgb_occurrences.json\').read_bytes()) if (RAW/\'sgb_occurrences.json\').exists() else {}\npoints=[];pointmeta=[]\nif geo_context.get(\'exceededTransferLimit\'):raise RuntimeError(\'SGB truncated\')\nfor feat in geo_context.get(\'features\',[]):\n if not feat.get(\'geometry\'):continue\n g=shape(feat[\'geometry\'])\n if g.geom_type!=\'Point\' or g.is_empty:continue\n points.append(transform(forward,g));pointmeta.append(feat[\'properties\'])\npt=STRtree(points) if points else None\npower=json.loads((RAW/\'sgb_power.json\').read_bytes()) if (RAW/\'sgb_power.json\').exists() else {};power_complete=bool(power.get(\'features\')) and not power.get(\'exceededTransferLimit\')\npowergeoms=[transform(forward,shape(f[\'geometry\'])) for f in power.get(\'features\',[]) if f.get(\'geometry\')] if power_complete else []\npower_tree=STRtree(powergeoms) if powergeoms else None\naudit=[];features=[]\nfor k,r in stock.items():\n row={k2:r[k2] for k2 in [\'process\',\'sople\',\'phase_name\',\'state\',\'city\',\'minerals\',\'age_days\',\'apt_conflict\']}\n row.update({\'geometry\':k in geoms,\'area_ha\':None,\'block_ha\':None,\'block_count\':None,\'occurrences_inside\':None,\'same_mineral_inside\':None,\'nearest_occurrence_km\':None,\'power_distance_km\':None})\n if k in geoms:\n  g=geoms[k];matches=bt.query(g,predicate=\'intersects\');ints=[g.intersection(blocks[int(i)]) for i in matches];blocked=unary_union(ints) if ints else None\n  blockha=blocked.area/10000 if blocked is not None else 0\n  ids=pt.query(g,predicate=\'covers\').tolist() if pt is not None else []\n  mset=set(r[\'minerals\'].split(\' | \'))-{\'\'};same=[i for i in ids if mset & minerals(pointmeta[i].get(\'SUBSTANCIAS\',\'\'))]\n  nearest=float(g.distance(points[int(pt.nearest(g))])/1000) if pt is not None else None\n  pd=float(g.distance(powergeoms[int(power_tree.nearest(g))])/1000) if power_tree is not None else None\n  row.update(metadata[k]);row.update({\'area_ha\':g.area/10000,\'block_ha\':blockha,\'block_count\':sum(x.area>100 for x in ints),\'occurrences_inside\':len(ids),\'same_mineral_inside\':len(same),\'nearest_occurrence_km\':nearest,\'power_distance_km\':pd,\'geological_ids\':[pointmeta[i].get(\'ID_OCORRENCIA\') for i in same],\'block_categories\':sorted({blockmeta[int(i)].get(\'CATEGORIA\',\'\') for i in matches if g.intersection(blocks[int(i)]).area>100})})\n  display=transform(reverse,g.simplify(100,preserve_topology=True));c=transform(reverse,g.representative_point())\n  row.update({\'lon\':c.x,\'lat\':c.y})\n  features.append({\'type\':\'Feature\',\'properties\':{\'process\':k,\'apt\':r[\'sople\']==\'Apta para Disponibilidade\',\'strategic\':bool(r[\'minerals\']),\'block\':blockha>.01,\'same_mineral\':len(same),\'area_ha\':round(row[\'area_ha\'],2)},\'geometry\':mapping(display)})\n audit.append(row)\nlookup={r[\'process\']:r for r in audit}\ngroups={\'union\':list(stock),\'scm_active\':[k for k,r in stock.items() if r[\'active\']==\'True\' and r[\'phase\'] in [\'8\',\'15\']],\'sople_apt\':[k for k,r in stock.items() if r[\'sople\']==\'Apta para Disponibilidade\'],\'strategic_apt\':[k for k,r in stock.items() if r[\'sople\']==\'Apta para Disponibilidade\' and r[\'minerals\']]}\nsummary={}\nfor label,keys in groups.items():\n gs=[geoms[k] for k in keys if k in geoms];u=unary_union(gs);idx=bt.query(u,predicate=\'intersects\');blockunion=unary_union([u.intersection(blocks[int(j)]) for j in idx]) if len(idx) else None\n summa=sum(g.area for g in gs)/10000;area=u.area/10000;over=blockunion.area/10000 if blockunion is not None else 0\n summary[label]={\'records\':len(keys),\'matched\':len(gs),\'missing\':len(keys)-len(gs),\'coverage\':len(gs)/len(keys),\'sum_process_ha\':summa,\'union_ha\':area,\'duplicate_coverage_ha\':summa-area,\'block_union_ha\':over,\'outside_block_layer_ha\':area-over,\'processes_block_overlap\':sum((lookup[k][\'block_ha\'] or 0)>.01 for k in keys),\'same_mineral_occurrence\':sum((lookup[k][\'same_mineral_inside\'] or 0)>0 for k in keys)}\n print(label,summary[label],flush=True)\nbymineral={}\nfor m in cfg[\'tokens\']:\n rr=[x for x in audit if x[\'sople\']==\'Apta para Disponibilidade\' and m in x[\'minerals\'].split(\' | \')]\n if not rr:continue\n bymineral[m]={\'records\':len(rr),\'matched\':sum(x[\'geometry\'] for x in rr),\'with_block\':sum((x[\'block_ha\'] or 0)>.01 for x in rr),\'with_same_mineral_evidence\':0,\'over730\':sum(int(x[\'age_days\'] or 0)>730 for x in rr),\'area_ha\':sum(x[\'area_ha\'] or 0 for x in rr)}\n # Match the specific mineral, not any other substance in a multi-mineral process.\n for x in rr:\n  if x[\'process\'] in geoms and pt is not None:\n   ii=pt.query(geoms[x[\'process\']],predicate=\'covers\')\n   bymineral[m][\'with_same_mineral_evidence\']+=int(any(m in minerals(pointmeta[int(i)].get(\'SUBSTANCIAS\',\'\')) for i in ii))\n# Spatial QA uses undensified geodesic polygon edges as an independent approximation.\nchecks=[]\nfor k in sorted(geoms)[::max(1,len(geoms)//100)][:100]:\n g=geoms[k];back=transform(reverse,g);ga=abs(geod.geometry_area_perimeter(shapely.orient_polygons(back))[0]);checks.append(abs(ga-g.area)/max(1,ga))\nreport={\'date\':\'2026-10-03\',\'sigmine_shape_date\':z.getinfo(\'BRASIL.shp\').date_time,\'block_shape_date\':zb.getinfo(\'BLOQUEIO.shp\').date_time,\'source_records\':len(s),\'block_features\':len(blocks),\'geological_points\':len(points),\'projection\':\'LAEA GRS80 centered at 15S 54W; area in hectares\',\'display_simplification_m\':100,\'invalid_repaired\':invalid,\'empty_excluded\':empty,\'duplicate_process_geometries\':sum(len(v)>1 for v in seen.values()),\'groups\':summary,\'minerals\':bymineral,\'power_layer_usable\':power_complete,\'qa_geodesic_relative_max\':max(checks,default=None),\'qa_geodesic_relative_median\':float(np.median(checks)) if checks else None,\'sources\':[{\'name\':p.name,\'bytes\':p.stat().st_size,\'sha256\':hashlib.sha256(p.read_bytes()).hexdigest()} for p in RAW.glob(\'*.zip\')],\'scope\':\'Spatial screening. Not certification of availability, reserves, legal impediment or infrastructure capacity.\'}\nassert all(v[\'union_ha\']<=v[\'sum_process_ha\']+1 and v[\'block_union_ha\']<=v[\'union_ha\']+1 for v in summary.values())\nsave(\'territorial_results.json\',report);save(\'territorial_audit.json\',audit);save(\'stock_polygons.geojson\',{\'type\':\'FeatureCollection\',\'features\':features})\nsave(\'stock_missing_geometry.json\',[r for r in audit if not r[\'geometry\']])\nprint(\'saved\',len(audit),len(features),flush=True)\n', 'statistical_expansion.py': "from pathlib import Path\nimport sys,csv,json,math,collections,hashlib\nimport numpy as np\nR=Path(__file__).resolve().parents[1];sys.path.append(str(R.parent/'tmp/science'))\nimport scipy,statsmodels.api as sm\nfrom scipy.stats import norm,rankdata\nfrom statsmodels.duration.hazard_regression import PHReg\nfrom statistics_pp import incidence\nD=R/'data/expansion';D.mkdir(exist_ok=True)\ndef save(n,x):(D/n).write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf8')\ndef csvsave(n,rows):\n with (D/n).open('w',encoding='utf-8-sig',newline='') as f:\n  w=csv.DictWriter(f,fieldnames=list(rows[0]),delimiter=';');w.writeheader();w.writerows(rows)\nrows=list(csv.DictReader((R/'data/complete/award_analysis.csv').open(encoding='utf-8-sig'),delimiter=';'))\nfor r in rows:\n for k in ['round','bid','ha','unpaid','strategic','time','cause','followup','cession','cessions']:r[k]=float(r[k])\n r['known']=r['substances'].strip().upper() not in ['','NAO SE APLICA','NÃO SE APLICA','NÃO INFORMADA','NÃO INFORMADO']\nregions={'Norte':{'Acre','Amapá','Amazonas','Pará','Rondônia','Roraima','Tocantins'},'Nordeste':{'Alagoas','Bahia','Ceará','Maranhão','Paraíba','Pernambuco','Piauí','Rio Grande do Norte','Sergipe'},'Centro-Oeste':{'Distrito Federal','Goiás','Mato Grosso','Mato Grosso do Sul'},'Sul':{'Paraná','Rio Grande do Sul','Santa Catarina'}}\nmr=[r for r in rows if r['round']!=1 and r['ha']>0 and r['known']]\nnames=['Intercepto','Log do lance','Log da área','Lavra','Proxy estratégica','Rodada 3','Rodada 4','Rodada 5','Rodada 8']+list(regions)\ndef xrow(r,rounds=True):return [1,math.log1p(r['bid']),math.log1p(r['ha']),int('Lavra' in r['regime']),r['strategic']]+([int(r['round']==k) for k in [3,4,5,8]] if rounds else [])+[int(r['state'] in s) for s in regions.values()]\nX=np.array([xrow(r) for r in mr]);y=np.array([r['unpaid'] for r in mr]);groups=np.array([r['origin'] for r in mr]);base=json.loads((R/'data/complete/results.json').read_bytes())\nfit=sm.GLM(y,X,family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':groups})\ndiff=float(max(abs(fit.params-np.array(base['logistic_region_known']['coef']))))\nprint('independent logit',diff,flush=True)\ndef metrics(y,p):\n ranks=rankdata(p);n1=sum(y);n0=len(y)-n1;auc=(sum(ranks[y==1])-n1*(n1+1)/2)/(n1*n0) if n1*n0 else None\n return {'n':len(y),'events':int(sum(y)),'observed':float(np.mean(y)),'predicted':float(np.mean(p)),'brier':float(np.mean((y-p)**2)),'auc':float(auc) if auc is not None else None,'log_loss':float(-np.mean(y*np.log(np.clip(p,1e-10,1))+(1-y)*np.log(np.clip(1-p,1e-10,1))))}\n# Origin-grouped folds prevent repeated origins leaking into training and testing.\nfold=np.array([int(hashlib.sha256(g.encode()).hexdigest()[:8],16)%5 for g in groups]);oof=np.zeros(len(mr));foldmetrics=[]\nfor k in range(5):\n tr=fold!=k;te=~tr;f=sm.GLM(y[tr],X[tr],family=sm.families.Binomial()).fit();oof[te]=f.predict(X[te]);foldmetrics.append(metrics(y[te],oof[te]))\ncal=[]\nfor ids in np.array_split(np.argsort(oof),10):cal.append({'n':len(ids),'predicted':float(np.mean(oof[ids])),'observed':float(np.mean(y[ids]))})\ncalfit=sm.GLM(y,sm.add_constant(np.log(np.clip(oof,1e-8,1-1e-8)/(1-np.clip(oof,1e-8,1-1e-8)))),family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':groups})\nTR=np.array([r['round']<8 for r in mr]);XT=np.array([xrow(r,False) for r in mr]);f=sm.GLM(y[TR],XT[TR],family=sm.families.Binomial()).fit();temporal=metrics(y[~TR],f.predict(XT[~TR]));temporal['training_n']=int(sum(TR));temporal['note']='Treino rodadas 2–5, teste rodada 8; especificação sem indicadores de rodada. Não mede inadimplência conciliada.'\n# Independent Cox/Breslow implementation under the original analysis population.\ncr=[r for r in rows if r['round']!=1 and r['ha']>0]\ncn=['Log do lance','Log da área','Lavra','Proxy estratégica','Rodada 3','Rodada 4','Rodada 5','Rodada 8']\nCX=np.array([[math.log1p(r['bid']),math.log1p(r['ha']),int('Lavra' in r['regime']),r['strategic'],*[int(r['round']==k) for k in [3,4,5,8]]] for r in cr]);tt=np.array([r['time'] for r in cr]);ee=np.array([r['cause']==1 for r in cr]);gg=np.array([r['origin'] for r in cr])\ncf=PHReg(tt,CX,status=ee,ties='breslow').fit();cdiff=float(max(abs(cf.params-np.array(base['cox_cession']['coef']))));print('independent Cox',cdiff,flush=True)\ninteractions=[]\nfor cutoff in [365,730]:\n xx=[];start=[];end=[];ev=[];cg=[]\n for i,r in enumerate(cr):\n  if tt[i]<=0:continue\n  xx.append([*CX[i],0,0]);start.append(0);end.append(min(tt[i],cutoff));ev.append(int(ee[i] and tt[i]<=cutoff));cg.append(gg[i])\n  if tt[i]>cutoff:\n   xx.append([*CX[i],CX[i,0],CX[i,3]]);start.append(cutoff);end.append(tt[i]);ev.append(int(ee[i]));cg.append(gg[i])\n tf=PHReg(np.array(end),np.array(xx),status=np.array(ev),entry=np.array(start),ties='breslow').fit()\n for j,label in [(-2,'Log do lance × período posterior'),(-1,'Proxy estratégica × período posterior')]:\n  interactions.append({'cutoff':cutoff,'variable':label,'ratio':float(np.exp(tf.params[j])),'lo':float(np.exp(tf.conf_int()[j,0])),'hi':float(np.exp(tf.conf_int()[j,1])),'p':float(tf.pvalues[j]),'covariance':'Informação do modelo; não robusta por origem. Contraste diagnóstico, não teste causal.'})\n# Family-wise Holm adjustment for the four prespecified diagnostic contrasts.\nix=np.argsort([a['p'] for a in interactions]);last=0\nfor j,k in enumerate(ix):last=max(last,min(1,(len(ix)-j)*interactions[k]['p']));interactions[k]['p_holm']=last\nprint('time interactions done',flush=True)\n# Bootstrap whole origins, including repeated allocations, not individual event rows.\nrng=np.random.default_rng(20261004);unique=np.unique([r['origin'] for r in rows]);index={g:[] for g in unique}\nfor i,r in enumerate(rows):index[r['origin']].append(i)\ntimes=np.array([int(r['time']) for r in rows]);causes=np.array([int(r['cause']) for r in rows]);boot=[]\nfor b in range(1000):\n ids=np.concatenate([index[g] for g in rng.choice(unique,len(unique),replace=True)]);z=incidence(times[ids],causes[ids],horizons=[365,730]);boot.append([h['incidence'] for h in z['horizons']])\nboot=np.array(boot);ci=[{'day':h,'estimate':incidence(times,causes,horizons=[h])['horizons'][0]['incidence'],'lo':float(np.quantile(boot[:,j],.025)),'hi':float(np.quantile(boot[:,j],.975)),'bootstrap_sd':float(np.std(boot[:,j],ddof=1))} for j,h in enumerate([365,730])]\n# A prospective, executable audit sample; truths intentionally remain blank.\nstock=json.loads((R/'site/inventory.json').read_bytes())\ndef sample_audit(pop,stratum,key,target,name):\n strata=collections.defaultdict(list)\n for r in pop:strata[stratum(r)].append(r)\n allocations={k:min(len(v),max(8,round(target*len(v)/len(pop)))) for k,v in strata.items()};out=[];summary=[]\n for k,rr in sorted(strata.items()):\n  nn=allocations[k];chosen=rng.choice(len(rr),nn,replace=False);summary.append({'stratum':k,'N':len(rr),'n':nn,'weight':len(rr)/nn})\n  for i in chosen:\n   r=rr[int(i)];out.append({'id':key(r),'stratum':k,'population_stratum':len(rr),'sample_stratum':nn,'weight':len(rr)/nn,'reviewer_1':'','reviewer_2':'','document_url':'','document_date':'','truth':'','decision':'','reason':'','status':'pendente'})\n csvsave(name,out);return {'population':len(pop),'sample':len(out),'strata':summary,'verified':0,'error_rate':None,'status':'Sorteio executado. Revisão documental humana ainda não realizada; nenhuma acurácia presumida.'}\nauditstock=sample_audit(stock,lambda r:('estrategico' if r['minerals'] else 'outro')+'|'+('conflito' if r['apt_conflict']=='True' else 'sem_conflito')+'|'+('apto' if r['sople']=='Apta para Disponibilidade' else 'somente_scm'),lambda r:r['process'],400,'audit_stock_sample.csv')\nauditcession=sample_audit(rows,lambda r:str(int(r['round']))+'|'+('cessao' if r['cession'] else 'sem_cessao'),lambda r:r['key'],300,'audit_cession_sample.csv')\n# Planning calculations, not a field experiment or causal estimate.\npower=[]\nfor p0 in [.05,.10,.17]:\n for rel in [.2,.3]:\n  p1=p0*(1-rel);pm=(p0+p1)/2;n=math.ceil(((norm.ppf(.975)*math.sqrt(2*pm*(1-pm))+norm.ppf(.8)*math.sqrt(p0*(1-p0)+p1*(1-p1)))/(p0-p1))**2)\n  for rho in [0,.01,.05]:\n   de=1+19*rho;power.append({'baseline':p0,'relative_reduction':rel,'p1':p1,'icc':rho,'cluster_size':20,'design_effect':de,'n_per_arm_independent':n,'n_per_arm_cluster_adjusted':math.ceil(n*de),'clusters_per_arm':math.ceil(n*de/20),'alpha':.05,'power':.8})\nreport={'versions':{'numpy':np.__version__,'scipy':scipy.__version__,'statsmodels':sm.__version__},'independent_logit_max_difference':diff,'independent_cox_max_difference':cdiff,'cross_validation':metrics(y,oof),'folds':foldmetrics,'calibration_deciles':cal,'calibration_intercept':float(calfit.params[0]),'calibration_slope':float(calfit.params[1]),'temporal_validation':temporal,'time_interactions':interactions,'bootstrap_replicates':1000,'bootstrap_origin_ci':ci,'stock_audit_design':auditstock,'cession_audit_design':auditcession,'pilot_power':power,'pilot_status':'Protocolo e dimensionamento executados. Não houve alteração de edital, tratamento de participantes ou experimento de campo.'}\nassert diff<1e-5 and cdiff<1e-5\nsave('statistics_advanced.json',report);print(json.dumps({k:report[k] for k in ['independent_logit_max_difference','independent_cox_max_difference','cross_validation','temporal_validation','time_interactions','bootstrap_origin_ci']},ensure_ascii=True),flush=True)\n", 'evaluate_audit.py': '"""Estimate stratified error only after all sampled units have adjudicated truth.\nUsage: python evaluate_audit.py audit_sample_reviewed.csv\ntruth: 1 classification confirmed, 0 classification incorrect.\nRequires both reviewers, source, reason, final decision and status=concluido.\nIncomplete and unresolved documents remain missing, never implicit successes.\n"""\nimport sys,csv,json,math,collections\ndef evaluate(path):\n rows=list(csv.DictReader(open(path,encoding=\'utf-8-sig\'),delimiter=\';\'))\n required=[\'reviewer_1\',\'reviewer_2\',\'document_url\',\'document_date\',\'truth\',\'decision\',\'reason\']\n pending=[r[\'id\'] for r in rows if r[\'status\']!=\'concluido\' or r[\'truth\'] not in [\'0\',\'1\'] or any(not r[k].strip() for k in required)]\n if pending:return {\'status\':\'incomplete\',\'sample\':len(rows),\'pending\':len(pending),\'error_rate\':None,\'interval\':None}\n groups=collections.defaultdict(list)\n for r in rows:groups[r[\'stratum\']].append(r)\n N=sum(int(rr[0][\'population_stratum\']) for rr in groups.values());estimate=variance=0\n for h,rr in groups.items():\n  Nh=int(rr[0][\'population_stratum\']);nh=len(rr);assert nh==int(rr[0][\'sample_stratum\'])\n  errors=[1-int(r[\'truth\']) for r in rr];p=sum(errors)/nh;estimate+=Nh/N*p\n  if nh>1:variance+=(Nh/N)**2*(1-nh/Nh)*sum((x-p)**2 for x in errors)/(nh-1)/nh\n  elif nh<Nh:raise ValueError(\'Non-census stratum requires at least two reviewed units\')\n se=math.sqrt(variance)\n return {\'status\':\'complete\',\'population\':N,\'sample\':len(rows),\'weighted_error\':estimate,\'standard_error\':se,\'normal_interval_95\':[max(0,estimate-1.96*se),min(1,estimate+1.96*se)],\'warning\':\'Normal approximation may be poor for sparse errors; report stratum counts and consider design-based alternatives. Zero observed errors does not prove zero population error.\'}\nif __name__==\'__main__\':print(json.dumps(evaluate(sys.argv[1]),ensure_ascii=False,indent=2))\n'}
for name, source in EXPANSION_SCRIPTS.items():
    (ROOT/"src"/name).write_text(source, encoding="utf8")
PACKED_INPUTS = '...'
for name, content in json.loads(gzip.decompress(base64.b64decode(PACKED_INPUTS))).items():
    p=ROOT/name
    p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(content,encoding="utf-8-sig" if p.suffix==".csv" else "utf8")


In [ ]:
GEO_MANIFEST = [{'file': 'SIGMINE_BRASIL.zip', 'url': 'https://dadosabertos.anm.gov.br/SIGMINE/PROCESSOS_MINERARIOS/BRASIL.zip', 'sha256': '1569f0ea6cc4b27f97eef32165d4d2ca703ee9d54d739db7416ee07528a562c6'}, {'file': 'BLOQUEIO.zip', 'url': 'https://dadosabertos.anm.gov.br/SIGMINE/BLOQUEIO.zip', 'sha256': '8c8a5a485b9ad35a849c3aa61a93efa61a3a1895e79c285e2d35b0be9e3a0507'}, {'file': 'sgb_occurrences.json', 'url': 'https://geoportal.sgb.gov.br/server/rest/services/geologia/ocorrencias/MapServer/0/query?where=1%3D1&outFields=OBJECTID%2CID_OCORRENCIA%2CSTATUS_ECONOMICO%2CSUBSTANCIAS%2CPROJETO%2CDATA_CADASTRO%2CMETODO_GEOPOSICIONAMENTO&returnGeometry=true&outSR=4674&f=geojson', 'sha256': 'ba5546c2fceff4677120e6d52d2de0d5f9d0bd923138ca78fb0e93a74b3d6da3'}]
PERMITIR_NOVA_EXTRAÇÃO_ESPACIAL = False
for item in GEO_MANIFEST:
    p=ROOT/"data/expansion/raw"/item["file"]
    p.parent.mkdir(parents=True,exist_ok=True)
    if not p.exists():
        req=urllib.request.Request(item["url"],headers={"User-Agent":"Mozilla/5.0"})
        with urllib.request.urlopen(req,timeout=240) as res, p.open("wb") as dst:
            import shutil
            shutil.copyfileobj(res,dst)
    equal=hashlib.sha256(p.read_bytes()).hexdigest()==item["sha256"]
    print(item["file"], "idêntico" if equal else "diferente")
    if not equal and not PERMITIR_NOVA_EXTRAÇÃO_ESPACIAL:
        raise RuntimeError("Fonte alterada. Use o arquivo preservado para reproduzir os resultados publicados.")


In [ ]:
for script in ["territorial_expansion.py","statistical_expansion.py"]:
    subprocess.run([sys.executable,str(ROOT/"src"/script)],cwd=ROOT,check=True)
print("Resultados em",ROOT/"data/expansion")


## 6. Revisão documental e uso responsável

Os arquivos audit_stock_sample.csv e audit_cession_sample.csv contêm o sorteio, os estratos e os pesos. Os campos de revisão ficam vazios. A rotina evaluate_audit.py recusa estimar erro enquanto a revisão estiver incompleta. Dois revisores devem registrar fonte, data, conclusão e justificativa.

A taxa de erro não foi medida. As ocorrências geológicas não certificam reservas. A ausência de bloqueio na camada não certifica área livre. O modelo financeiro tem desempenho temporal limitado. Os cálculos de poder dimensionam hipóteses de um piloto ainda não executado.

In [ ]:
subprocess.run([sys.executable,str(ROOT/"src/evaluate_audit.py"),str(ROOT/"data/expansion/audit_stock_sample.csv")],check=True)
